# TP 1: LDA/QDA y optimización matemática de modelos

## Integrantes

- Nicolás Rodriguez da Cruz
- Francisco Cofré
- Gaspar Acevedo Zain
- Juan Chunga
- Rodrigo Nicolás Lauro

## Versiones de Librerías utilizadas

- `Python`: versión 3.11.9
- `numpy`: versión 2.2.0
- `scipy`: versión 1.15.3

# Intro teórica + Código provisto + Consigna QDA

## Intro teórica

### Definición: Clasificador Bayesiano

Sean $k$ poblaciones, $x \in \mathbb{R}^p$ puede pertenecer a cualquiera $g \in \mathcal{G}$ de ellas. Bajo un esquema bayesiano, se define entonces $\pi_j \doteq P(G = j)$ la probabilidad *a priori* de que $X$ pertenezca a la clase *j*, y se **asume conocida** la distribución condicional de cada observable dado su clase $f_j \doteq f_{X|G=j}$.

De esta manera dicha probabilidad *a posteriori* resulta
$$
P(G|_{X=x} = j) = \frac{f_{X|G=j}(x) \cdot p_G(j)}{f_X(x)} \propto f_j(x) \cdot \pi_j
$$

La regla de decisión de Bayes es entonces
$$
H(x) \doteq \arg \max_{g \in \mathcal{G}} \{ P(G|_{X=x} = j) \} = \arg \max_{g \in \mathcal{G}} \{ f_j(x) \cdot \pi_j \}
$$

es decir, se predice a $x$ como perteneciente a la población $j$ cuya probabilidad a posteriori es máxima.

*Ojo, a no desesperar! $\pi_j$ no es otra cosa que una constante prefijada, y $f_j$ es, en su esencia, un campo escalar de $x$ a simplemente evaluar.*

### Distribución condicional

Para los clasificadores de discriminante cuadrático y lineal (QDA/LDA) se asume que $X|_{G=j} \sim \mathcal{N}_p(\mu_j, \Sigma_j)$, es decir, se asume que cada población sigue una distribución normal.

Por definición, se tiene entonces que para una clase $j$:
$$
f_j(x) = \frac{1}{(2 \pi)^\frac{p}{2} \cdot |\Sigma_j|^\frac{1}{2}} e^{- \frac{1}{2}(x-\mu_j)^T \Sigma_j^{-1} (x- \mu_j)}
$$

Aplicando logaritmo (que al ser una función estrictamente creciente no afecta el cálculo de máximos/mínimos), queda algo mucho más práctico de trabajar:

$$
\log{f_j(x)} = -\frac{1}{2}\log |\Sigma_j| - \frac{1}{2} (x-\mu_j)^T \Sigma_j^{-1} (x- \mu_j) + C
$$

Observar que en este caso $C=-\frac{p}{2} \log(2\pi)$, pero no se tiene en cuenta ya que al tener una constante aditiva en todas las clases, no afecta al cálculo del máximo.

### LDA

En el caso de LDA se hace una suposición extra, que es $X|_{G=j} \sim \mathcal{N}_p(\mu_j, \Sigma)$, es decir que las poblaciones no sólo siguen una distribución normal sino que son de igual matriz de covarianzas. Reemplazando arriba se obtiene entonces:

$$
\log{f_j(x)} =  -\frac{1}{2}\log |\Sigma| - \frac{1}{2} (x-\mu_j)^T \Sigma^{-1} (x- \mu_j) + C
$$

Ahora, como $-\frac{1}{2}\log |\Sigma|$ es común a todas las clases se puede incorporar a la constante aditiva y, distribuyendo y reagrupando términos sobre $(x-\mu_j)^T \Sigma^{-1} (x- \mu_j)$ se obtiene finalmente:

$$
\log{f_j(x)} =  \mu_j^T \Sigma^{-1} (x- \frac{1}{2} \mu_j) + C'
$$

### Entrenamiento/Ajuste

Obsérvese que para ambos modelos, ajustarlos a los datos implica estimar los parámetros $(\mu_j, \Sigma_j) \; \forall j = 1, \dots, k$ en el caso de QDA, y $(\mu_j, \Sigma)$ para LDA.

Estos parámetros se estiman por máxima verosimilitud, de manera que los estimadores resultan:

* $\hat{\mu}_j = \bar{x}_j$ el promedio de los $x$ de la clase *j*
* $\hat{\Sigma}_j = s^2_j$ la matriz de covarianzas estimada para cada clase *j*
* $\hat{\pi}_j = f_{R_j} = \frac{n_j}{n}$ la frecuencia relativa de la clase *j* en la muestra
* $\hat{\Sigma} = \frac{1}{n} \sum_{j=1}^k n_j \cdot s^2_j$ el promedio ponderado (por frecs. relativas) de las matrices de covarianzas de todas las clases. *Observar que se utiliza el estimador de MV y no el insesgado*

Es importante notar que si bien todos los $\mu, \Sigma$ deben ser estimados, la distribución *a priori* puede no inferirse de los datos sino asumirse previamente, utilizándose como entrada del modelo.

### Predicción

Para estos modelos, al igual que para cualquier clasificador Bayesiano del tipo antes visto, la estimación de la clase es por método *plug-in* sobre la regla de decisión $H(x)$, es decir devolver la clase que maximiza $\hat{f}_j(x) \cdot \hat{\pi}_j$, o lo que es lo mismo $\log\hat{f}_j(x) + \log\hat{\pi}_j$.

## Código provisto

Con el fin de no retrasar al alumno con cuestiones estructurales y/o secundarias al tema que se pretende tratar, se provee una base de código que **no es obligatoria de usar** pero se asume que resulta resulta beneficiosa.

In [2]:
import numpy as np
import pandas as pd
import numpy.linalg as LA
from scipy.linalg import cholesky, solve_triangular
from scipy.linalg.lapack import dtrtri

### Base code

In [3]:
class BaseBayesianClassifier:
  def __init__(self):
    pass

  def _estimate_a_priori(self, y):
    a_priori = np.bincount(y.flatten().astype(int)) / y.size
    # Q3: para que sirve bincount?
    return np.log(a_priori)

  def _fit_params(self, X, y):
    # estimate all needed parameters for given model
    raise NotImplementedError()

  def _predict_log_conditional(self, x, class_idx):
    # predict the log(P(x|G=class_idx)), the log of the conditional probability of x given the class
    # this should depend on the model used
    raise NotImplementedError()

  def fit(self, X, y, a_priori=None):
    # if it's needed, estimate a priori probabilities
    self.log_a_priori = self._estimate_a_priori(y) if a_priori is None else np.log(a_priori)

    # now that everything else is in place, estimate all needed parameters for given model
    self._fit_params(X, y)
    # Q4: por que el _fit_params va al final? no se puede mover a, por ejemplo, antes de la priori?

  def predict(self, X):
    # this is actually an individual prediction encased in a for-loop
    m_obs = X.shape[1]
    y_hat = np.empty(m_obs, dtype=int)

    for i in range(m_obs):
      y_hat[i] = self._predict_one(X[:,i].reshape(-1,1))

    # return prediction as a row vector (matching y)
    return y_hat.reshape(1,-1)

  def _predict_one(self, x):
    # calculate all log posteriori probabilities (actually, +C)
    log_posteriori = [ log_a_priori_i + self._predict_log_conditional(x, idx) for idx, log_a_priori_i
                  in enumerate(self.log_a_priori) ]

    # return the class that has maximum a posteriori probability
    return np.argmax(log_posteriori)

In [4]:
class QDA(BaseBayesianClassifier):

  def _fit_params(self, X, y):
    # estimate each covariance matrix
    self.inv_covs = [LA.inv(np.cov(X[:,y.flatten()==idx], bias=True))
                      for idx in range(len(self.log_a_priori))]
    # Q5: por que hace falta el flatten y no se puede directamente X[:,y==idx]?
    # Q6: por que se usa bias=True en vez del default bias=False?
    self.means = [X[:,y.flatten()==idx].mean(axis=1, keepdims=True)
                  for idx in range(len(self.log_a_priori))]
    # Q7: que hace axis=1? por que no axis=0?

  def _predict_log_conditional(self, x, class_idx):
    # predict the log(P(x|G=class_idx)), the log of the conditional probability of x given the class
    # this should depend on the model used
    inv_cov = self.inv_covs[class_idx]
    unbiased_x =  x - self.means[class_idx]
    return 0.5*np.log(LA.det(inv_cov)) -0.5 * unbiased_x.T @ inv_cov @ unbiased_x

In [5]:
class TensorizedQDA(QDA):

    def _fit_params(self, X, y):
        # ask plain QDA to fit params
        super()._fit_params(X,y)

        # stack onto new dimension
        self.tensor_inv_cov = np.stack(self.inv_covs)
        self.tensor_means = np.stack(self.means)

    def _predict_log_conditionals(self,x):
        unbiased_x = x - self.tensor_means
        inner_prod = unbiased_x.transpose(0,2,1) @ self.tensor_inv_cov @ unbiased_x

        return 0.5*np.log(LA.det(self.tensor_inv_cov)) - 0.5 * inner_prod.flatten()

    def _predict_one(self, x):
        # return the class that has maximum a posteriori probability
        return np.argmax(self.log_a_priori + self._predict_log_conditionals(x))

In [6]:
class QDA_Chol1(BaseBayesianClassifier):
  def _fit_params(self, X, y):
    self.L_invs = [
        LA.inv(cholesky(np.cov(X[:,y.flatten()==idx], bias=True), lower=True))
        for idx in range(len(self.log_a_priori))
    ]

    self.means = [X[:,y.flatten()==idx].mean(axis=1, keepdims=True)
                  for idx in range(len(self.log_a_priori))]

  def _predict_log_conditional(self, x, class_idx):
    L_inv = self.L_invs[class_idx]
    unbiased_x =  x - self.means[class_idx]

    y = L_inv @ unbiased_x

    return np.log(L_inv.diagonal().prod()) -0.5 * (y**2).sum()

In [7]:
class QDA_Chol2(BaseBayesianClassifier):
  def _fit_params(self, X, y):
    self.Ls = [
        cholesky(np.cov(X[:,y.flatten()==idx], bias=True), lower=True)
        for idx in range(len(self.log_a_priori))
    ]

    self.means = [X[:,y.flatten()==idx].mean(axis=1, keepdims=True)
                  for idx in range(len(self.log_a_priori))]

  def _predict_log_conditional(self, x, class_idx):
    L = self.Ls[class_idx]
    unbiased_x =  x - self.means[class_idx]

    y = solve_triangular(L, unbiased_x, lower=True)

    return -np.log(L.diagonal().prod()) -0.5 * (y**2).sum()

In [8]:
class QDA_Chol3(BaseBayesianClassifier):
  def _fit_params(self, X, y):
    self.L_invs = [
        dtrtri(cholesky(np.cov(X[:,y.flatten()==idx], bias=True), lower=True), lower=1)[0]
        for idx in range(len(self.log_a_priori))
    ]

    self.means = [X[:,y.flatten()==idx].mean(axis=1, keepdims=True)
                  for idx in range(len(self.log_a_priori))]

  def _predict_log_conditional(self, x, class_idx):
    L_inv = self.L_invs[class_idx]
    unbiased_x =  x - self.means[class_idx]

    y = L_inv @ unbiased_x

    return np.log(L_inv.diagonal().prod()) -0.5 * (y**2).sum()

### Datasets

Observar que se proveen **4 datasets diferentes**, el código de ejemplo usa uno solo pero eso no significa que ustedes se limiten al mismo. También pueden usar otros datasets de su elección.

In [9]:
from sklearn.datasets import load_iris, fetch_openml, load_wine
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

def get_iris_dataset():
  data = load_iris()
  X_full = data.data
  y_full = np.array([data.target_names[y] for y in data.target.reshape(-1,1)])
  return X_full, y_full

def get_penguins_dataset():
    # get data
    df, tgt = fetch_openml(name="penguins", return_X_y=True, as_frame=True, parser='auto')

    # drop non-numeric columns
    df.drop(columns=["island","sex"], inplace=True)

    # drop rows with missing values
    mask = df.isna().sum(axis=1) == 0
    df = df[mask]
    tgt = tgt[mask]

    return df.values, tgt.to_numpy().reshape(-1,1)

def get_wine_dataset():
    # get data
    data = load_wine()
    X_full = data.data
    y_full = np.array([data.target_names[y] for y in data.target.reshape(-1,1)])
    return X_full, y_full

def get_letters_dataset():
    # get data
    letter = fetch_openml('letter', version=1, as_frame=False)
    return letter.data, letter.target.reshape(-1,1)

def label_encode(y_full):
    return LabelEncoder().fit_transform(y_full.flatten()).reshape(y_full.shape)

def split_transpose(X, y, test_size, random_state):
    # X_train, X_test, y_train, y_test but all transposed
    return [elem.T for elem in train_test_split(X, y, test_size=test_size, random_state=random_state)]

### Benchmarking

Nota: esta clase fue creada bastante rápido y no pretende ser una plataforma súper confiable sobre la que basarse, sino más bien una herramienta simple con la que poder medir varios runs y agregar la información.

En forma rápida, `warmup` es la cantidad de runs para warmup, `mem_runs` es la cantidad de runs en las que se mide el pico de uso de RAM y `n_runs` es la cantidad de runs en las que se miden tiempos.

La razón por la que se separan es que medir memoria hace ~2.5x más lento cada run, pero al mismo tiempo se estabiliza mucho más rápido.

**Importante:** tener en cuenta que los modelos que predicen en batch (usan `predict` directamente) deberían consumir, como mínimo, $n$ veces la memoria de los que predicen por observación.

In [10]:
import time
from tqdm.notebook import tqdm
from numpy.random import RandomState
import tracemalloc

RNG_SEED = 6553

class Benchmark:
    def __init__(self, X, y, n_runs=1000, warmup=100, mem_runs=100, test_sz=0.3, rng_seed=RNG_SEED, same_splits=True):
        self.X = X
        self.y = y
        self.n = n_runs
        self.warmup = warmup
        self.mem_runs = mem_runs
        self.test_sz = test_sz
        self.det = same_splits
        if self.det:
            self.rng_seed = rng_seed
        else:
            self.rng = RandomState(rng_seed)

        self.data = dict()

        print("Benching params:")
        print("Total runs:",self.warmup+self.mem_runs+self.n)
        print("Warmup runs:",self.warmup)
        print("Peak Memory usage runs:", self.mem_runs)
        print("Running time runs:", self.n)
        approx_test_sz = int(self.y.size * self.test_sz)
        print("Train size rows (approx):",self.y.size - approx_test_sz)
        print("Test size rows (approx):",approx_test_sz)
        print("Test size fraction:",self.test_sz)

    def bench(self, model_class, **kwargs):
        name = model_class.__name__
        time_data = np.empty((self.n, 3), dtype=float)  # train_time, test_time, accuracy
        mem_data = np.empty((self.mem_runs, 2), dtype=float)  # train_peak_mem, test_peak_mem
        rng = RandomState(self.rng_seed) if self.det else self.rng


        for i in range(self.warmup):
            # Instantiate model with error check for unsupported parameters
            model = model_class(**kwargs)

            # Generate current train-test split
            X_train, X_test, y_train, y_test = split_transpose(
                self.X, self.y,
                test_size=self.test_sz,
                random_state=rng
            )
            # Run training and prediction (timing or memory measurement not recorded)
            model.fit(X_train, y_train)
            model.predict(X_test)

        for i in tqdm(range(self.mem_runs), total=self.mem_runs, desc=f"{name} (MEM)"):

            model = model_class(**kwargs)

            X_train, X_test, y_train, y_test = split_transpose(
                self.X, self.y,
                test_size=self.test_sz,
                random_state=rng
            )

            tracemalloc.start()

            t1 = time.perf_counter()
            model.fit(X_train, y_train)
            t2 = time.perf_counter()

            _, train_peak = tracemalloc.get_traced_memory()
            tracemalloc.reset_peak()

            model.predict(X_test)
            t3 = time.perf_counter()
            _, test_peak = tracemalloc.get_traced_memory()
            tracemalloc.stop()

            mem_data[i,] = (
                train_peak / (1024 * 1024),
                test_peak / (1024 * 1024)
            )

        for i in tqdm(range(self.n), total=self.n, desc=f"{name} (TIME)"):
            model = model_class(**kwargs)

            X_train, X_test, y_train, y_test = split_transpose(
                self.X, self.y,
                test_size=self.test_sz,
                random_state=rng
            )

            t1 = time.perf_counter()
            model.fit(X_train, y_train)
            t2 = time.perf_counter()
            preds = model.predict(X_test)
            t3 = time.perf_counter()

            time_data[i,] = (
                (t2 - t1) * 1000,
                (t3 - t2) * 1000,
                (y_test.flatten() == preds.flatten()).mean()
            )

        self.data[name] = (time_data, mem_data)

        # Devolvemos el modelo para facilitar el análisis de ciertas properties del mismo
        return model

    def summary(self, baseline=None):
        aux = []
        for name, (time_data, mem_data) in self.data.items():
            result = {
                'model': name,
                'train_median_ms': np.median(time_data[:, 0]),
                'train_std_ms': time_data[:, 0].std(),
                'test_median_ms': np.median(time_data[:, 1]),
                'test_std_ms': time_data[:, 1].std(),
                'mean_accuracy': time_data[:, 2].mean(),
                'train_mem_median_mb': np.median(mem_data[:, 0]),
                'train_mem_std_mb': mem_data[:, 0].std(),
                'test_mem_median_mb': np.median(mem_data[:, 1]),
                'test_mem_std_mb': mem_data[:, 1].std()
            }
            aux.append(result)
        df = pd.DataFrame(aux).set_index('model')

        if baseline is not None and baseline in self.data:
            df['train_speedup'] = df.loc[baseline, 'train_median_ms'] / df['train_median_ms']
            df['test_speedup'] = df.loc[baseline, 'test_median_ms'] / df['test_median_ms']
            df['train_mem_reduction'] = df.loc[baseline, 'train_mem_median_mb'] / df['train_mem_median_mb']
            df['test_mem_reduction'] = df.loc[baseline, 'test_mem_median_mb'] / df['test_mem_median_mb']
        return df

### Ejemplo

In [11]:
# levantamos el dataset Wine, que tiene 13 features y 178 observaciones en total
X_full, y_full = get_wine_dataset()

X_full.shape, y_full.shape

((178, 13), (178, 1))

In [12]:
# encodeamos a número las clases
y_full_encoded = label_encode(y_full)

y_full[:5], y_full_encoded[:5]

(array([['class_0'],
        ['class_0'],
        ['class_0'],
        ['class_0'],
        ['class_0']], dtype='<U7'),
 array([[0],
        [0],
        [0],
        [0],
        [0]]))

In [13]:
# generamos el benchmark
# observar que son valores muy bajos de runs para que corra rápido ahora
b = Benchmark(
    X_full, y_full_encoded,
    n_runs = 100,
    warmup = 20,
    mem_runs = 20,
    test_sz = 0.3,
    same_splits = False
)

Benching params:
Total runs: 140
Warmup runs: 20
Peak Memory usage runs: 20
Running time runs: 100
Train size rows (approx): 125
Test size rows (approx): 53
Test size fraction: 0.3


In [14]:
# bencheamos un par
to_bench = [QDA]

for model in to_bench:
    b.bench(model)

QDA (MEM):   0%|          | 0/20 [00:00<?, ?it/s]

QDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

In [15]:
# como es una clase, podemos seguir bencheando más después
b.bench(TensorizedQDA)

TensorizedQDA (MEM):   0%|          | 0/20 [00:00<?, ?it/s]

TensorizedQDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

In [16]:
# hacemos un summary
b.summary()

,train_median_ms,train_std_ms,test_median_ms,test_std_ms,mean_accuracy,train_mem_median_mb,train_mem_std_mb,test_mem_median_mb,test_mem_std_mb
model,,,,,,,,,
QDA,0.5846,0.364274,4.58330,1.393453,0.982407,0.019039,0.063834,0.008064,0.061007
TensorizedQDA,0.5045,1.708979,1.71115,1.414832,0.982593,0.018593,0.042113,0.012001,0.042387


In [17]:
# son muchos datos! nos quedamos con un par nomás
summ = b.summary()

# como es un pandas DataFrame, subseteamos columnas fácil
summ[['train_median_ms', 'test_median_ms','mean_accuracy']]

,train_median_ms,test_median_ms,mean_accuracy
model,,,
QDA,0.5846,4.58330,0.982407
TensorizedQDA,0.5045,1.71115,0.982593


In [18]:
# podemos setear un baseline para que fabrique columnas de comparación
summ = b.summary(baseline='QDA')

summ

,train_median_ms,train_std_ms,test_median_ms,test_std_ms,mean_accuracy,train_mem_median_mb,train_mem_std_mb,test_mem_median_mb,test_mem_std_mb,train_speedup,test_speedup,train_mem_reduction,test_mem_reduction
model,,,,,,,,,,,,,
QDA,0.5846,0.364274,4.58330,1.393453,0.982407,0.019039,0.063834,0.008064,0.061007,1.000000,1.000000,1.000000,1.000000
TensorizedQDA,0.5045,1.708979,1.71115,1.414832,0.982593,0.018593,0.042113,0.012001,0.042387,1.158771,2.678491,1.024005,0.671925


In [19]:
summ[[
    'train_median_ms', 'test_median_ms','mean_accuracy',
    'train_speedup', 'test_speedup',
    'train_mem_reduction', 'test_mem_reduction'
]]

,train_median_ms,test_median_ms,mean_accuracy,train_speedup,test_speedup,train_mem_reduction,test_mem_reduction
model,,,,,,,
QDA,0.5846,4.58330,0.982407,1.000000,1.000000,1.000000,1.000000
TensorizedQDA,0.5045,1.71115,0.982593,1.158771,2.678491,1.024005,0.671925


## Consigna QDA

**Notación**: en general notamos

* $k$ la cantidad de clases
* $n$ la cantidad de observaciones
* $p$ la cantidad de features/variables/predictores

**Sugerencia:** combinaciones adecuadas de `transpose`, `stack`, `reshape` y, ocasionalmente, `flatten` y `diagonal` suele ser más que suficiente. Se recomienda *fuertemente* explorar la dimensionalidad de cada elemento antes de implementar las clases.

### Tensorización

En esta sección nos vamos a ocupar de hacer que el modelo sea más rápido para generar predicciones, observando que incurre en un doble `for` dado que predice en forma individual un escalar para cada observación, para cada clase. Paralelizar ambos vía tensorización suena como una gran vía de mejora de tiempos.

#### 1) Diferencias entre `QDA`y `TensorizedQDA`

1. ¿Sobre qué paraleliza `TensorizedQDA`? ¿Sobre las $k$ clases, las $n$ observaciones a predecir, o ambas?
2. Analizar los shapes de `tensor_inv_covs` y `tensor_means` y explicar paso a paso cómo es que `TensorizedQDA` llega a predecir lo mismo que `QDA`.

#### 2) Optimización

Debido a la forma cuadrática de QDA, no se puede predecir para $n$ observaciones en una sola pasada (utilizar $X \in \mathbb{R}^{p \times n}$ en vez de $x \in \mathbb{R}^p$) sin pasar por una matriz de $n \times n$ en donde se computan todas las interacciones entre observaciones. Se puede acceder al resultado recuperando sólo la diagonal de dicha matriz, pero resulta ineficiente en tiempo y (especialmente) en memoria. Aún así, es *posible* que el modelo funcione más rápido.

3. Implementar el modelo `FasterQDA` (se recomienda heredarlo de `TensorizedQDA`) de manera de eliminar el ciclo for en el método predict.
4. Mostrar dónde aparece la mencionada matriz de $n \times n$, donde $n$ es la cantidad de observaciones a predecir.
5. Demostrar que
$$
diag(A \cdot B) = \sum_{cols} A \odot B^T = np.sum(A \odot B^T, axis=1)
$$ es decir, que se puede "esquivar" la matriz de $n \times n$ usando matrices de $n \times p$. También se puede usar, de forma equivalente,
$$
np.sum(A^T \odot B, axis=0).T
$$
queda a preferencia del alumno cuál usar.
6. Utilizar la propiedad antes demostrada para reimplementar la predicción del modelo `FasterQDA` de forma eficiente en un nuevo modelo `EfficientQDA`.
7. Comparar la performance de las 4 variantes de QDA implementadas hasta ahora (no Cholesky) ¿Qué se observa? A modo de opinión ¿Se condice con lo esperado?

### Cholesky

Hasta ahora todos los esfuerzos fueron enfocados en realizar una predicción más rápida. Los tiempos de entrenamiento (teóricos al menos) siguen siendo los mismos o hasta (minúsculamente) peores, dado que todas las mejoras siguen llamando al método `_fit_params` original de `QDA`.

La descomposición/factorización de [Cholesky](https://en.wikipedia.org/wiki/Cholesky_decomposition#Statement) permite factorizar una matriz definida positiva $A = LL^T$ donde $L$ es una matriz triangular inferior. En particular, si bien se asume que $p \ll n$, invertir la matriz de covarianzas $\Sigma$ para cada clase impone un cuello de botella que podría alivianarse. Teniendo en cuenta que las matrices de covarianza son simétricas y salvo degeneración, definidas positivas, Cholesky como mínimo debería permitir invertir la matriz más rápido.

*Nota: observar que calcular* $A^{-1}b$ *equivale a resolver el sistema* $Ax=b$.

#### 3) Diferencias entre implementaciones de `QDA_Chol`

8. Si una matriz $A$ tiene fact. de Cholesky $A=LL^T$, expresar $A^{-1}$ en términos de $L$. ¿Cómo podría esto ser útil en la forma cuadrática de QDA?
7. Explicar las diferencias entre `QDA_Chol1`y `QDA` y cómo `QDA_Chol1` llega, paso a paso, hasta las predicciones.
8. ¿Cuáles son las diferencias entre `QDA_Chol1`, `QDA_Chol2` y `QDA_Chol3`?
9. Comparar la performance de las 7 variantes de QDA implementadas hasta ahora ¿Qué se observa?¿Hay alguna de las implementaciones de `QDA_Chol` que sea claramente mejor que las demás?¿Alguna que sea peor?

#### 4) Optimización

12. Implementar el modelo `TensorizedChol` paralelizando sobre clases/observaciones según corresponda. Se recomienda heredarlo de alguna de las implementaciones de `QDA_Chol`, aunque la elección de cuál de ellas queda a cargo del alumno según lo observado en los benchmarks de puntos anteriores.
13. Implementar el modelo `EfficientChol` combinando los insights de `EfficientQDA` y `TensorizedChol`. Si se desea, se puede implementar `FasterChol` como ayuda, pero no se contempla para el punto.
13. Comparar la performance de las 9 variantes de QDA implementadas ¿Qué se observa? A modo de opinión ¿Se condice con lo esperado?

### Importante:

Las métricas que se observan al realizar benchmarking son muy dependientes del código que se ejecuta, y por tanto de las versiones de las librerías utilizadas. Una forma de unificar esto es utilizando un gestor de versiones y paquetes como _uv_ o _Poetry_, otra es simplemente usando una misma VM como la que provee Colab.

**Cada equipo debe informar las versiones de Python, NumPy y SciPy con que fueron obtenidos los resultados. En caso de que sean múltiples, agregar todos los casos**. La siguiente celda provee una ayuda para hacerlo desde un notebook, aunque como es una secuencia de comandos también sirve para consola.

In [ ]:
%%bash
python --version
pip freeze | grep -E "scipy|numpy"

**Comentario:** yo utilicé los siguientes parámetros para mi run de prueba. Esto NO significa que ustedes tengan que usar los mismos, tampoco el mismo dataset. Se agregó al notebook simplemente porque fue una pregunta común en cohortes anteriores.

In [101]:
# dataset de letters
X_letter, y_letter = get_letters_dataset()

# encoding de labels
y_letter_encoded = label_encode(y_letter.reshape(-1,1))

# instanciacion del benchmark
b = Benchmark(
    X_letter, y_letter_encoded,
    same_splits=False,
    n_runs=100,
    warmup=20,
    mem_runs=30,
    test_sz=0.2
)

Benching params:
Total runs: 150
Warmup runs: 20
Peak Memory usage runs: 30
Running time runs: 100
Train size rows (approx): 16000
Test size rows (approx): 4000
Test size fraction: 0.2


# Resolución del Trabajo Práctico número 1

## Carga de Datasets

In [ ]:
RNG_SEED = 6553

In [ ]:
# Wine Dataset

X_full_wine, y_full_wine = get_wine_dataset()
y_full_encoded_wine = label_encode(y_full_wine)

In [ ]:
# Iris Dataset

X_full_iris, y_full_iris = get_iris_dataset()
y_full_encoded_iris = label_encode(y_full_iris)

In [22]:
# Penguins Dataset

X_full_penguins, y_full_penguins = get_penguins_dataset()
y_full_encoded_penguins = label_encode(y_full_penguins)

In [ ]:
# Letters Dataset

X_full_letters, y_full_letters = get_letters_dataset()
y_full_encoded_letters = label_encode(y_full_letters)

## Punto 1. Diferencias entre `QDA`y `TensorizedQDA`

1. ¿Sobre qué paraleliza `TensorizedQDA`? ¿Sobre las $k$ clases, las $n$ observaciones a predecir, o ambas?
2. Analizar los shapes de `tensor_inv_covs` y `tensor_means` y explicar paso a paso cómo es que `TensorizedQDA` llega a predecir lo mismo que `QDA`.

### Respuestas del Punto 1

#### Respuesta Punto 1.1 - ¿Sobre qué paraleliza `TensorizedQDA`? ¿Sobre las $k$ clases, las $n$ observaciones a predecir, o ambas?

`TensorizedQDA` paraleliza sobre las $k$ clases, pero no sobre las $n$ observaciones a predecir. Esto lo podemos notar en las siguientes instrucciones de código:

- Paralelización sobre $k$ clases
  - `QDA` hereda e implementa el método `_predict_one` de la clase `BaseBayesianClassifier`, en donde se llama al método `_predict_log_conditional` pasándole el la observación `x` y el índice de la clase `class_idx`
    - Es decir, al pasarle un valor de `class_idx` implica que se realiza la predicción de $log(P(x|G=class_idx)$ para cada observación en cada clase por separado.
  - `TensorizedQDA` tiene su propia implementación del método `_predict_one`, en donde solamente pasa la observación `x` al método `_predict_log_conditionals`, en donde se utilizan los attributes de `tensor_inv_covs` y `tensor_means` para paralelizar sobre las tres clases.
    - Es decir, se puede calcular la predicción de $log(P(x|G=class_idx)$ en las tres clases en paralelo al solamente pasar la observación `x`.

- ¿Cómo se procesan las $n$ observaciones?
  - Podemos observar que tanto `QDA` como `TensorizedQDA` hacen uso del método `predict` heredado de la clase `BaseBayesianClassifier`, en donde se implementa un ciclo `for` sobre cada observación del dataset.
  - Por este motivo es que podemos decir que las $n$ observaciones ***NO*** son paralelizadas en `TensorizedQDA`

#### Respuesta  Punto 1.2 - Analizar los shapes de `tensor_inv_covs` y `tensor_means` y explicar paso a paso cómo es que `TensorizedQDA` llega a predecir lo mismo que `QDA`.

NOTA: Para analizar los shapes de `tensor_inv_covs` y `tensor_means`, lo que hicimos fue modificar el método `bench` de la clase `Benchmark` para que devuelva el modelo sobre el cuál se está trabajando.
De esta manera tenemos una forma fácil de analizar cada atributo que nos interese, a la vez que hacemos un benchmark.

`tensor_inv_covs` y `tensor_means` son ambos arrays n-dimensionales de numpy (`numpy.ndarray`) con las siguientes características:
- `tensor_inv_covs`
    - Tiene un shape de `(3, 13, 13)` para el `wine` dataset, es decir, tiene 3 valores que corresponden a las `3 clases` del dataset.
    - Luego, en cada valor de clase, tienen los `13 attributes` del mismo dataset, y en cada uno de ellos se guardan las covarianzas con el resto de los atributos (12 valores + la covarianza consigo misma, es decir, la varianza de dicho atributo).
    - Corresponde al término $\hat{\Sigma}_j = s^2_j$ (o la matriz de covarianzas estimada para cada clase *j*) de la siguiente fórmula:
        - $\log{f_j(x)} = -\frac{1}{2}\log |\Sigma_j| - \frac{1}{2} (x-\mu_j)^T \Sigma_j^{-1} (x- \mu_j) + C$
- `tensor_means`
    - Tiene un shape de `(3, 13, 1)` para el `wine` dataset, es decir, tienen 3 valores que corresponden a las `3 clases` del dataset.
    - Luego, en cada valor de clase, tienen los `13 attributes` del mismo dataset, y en cada uno de ellos se guarda el promedio de valores para una clase y atributo determinado.
    - Corresponde al término $\hat{\mu}_j = \bar{x}_j$ (o el promedio de los $x$ de la clase *j*) de la siguiente fórmula:
        - $\log{f_j(x)} = -\frac{1}{2}\log |\Sigma_j| - \frac{1}{2} (x-\mu_j)^T \Sigma_j^{-1} (x- \mu_j) + C$

`TensorizedQDA` tiene sus propias implementaciones de los métodos `_fit_params`, `_predict_log_conditionals`, y `_predict_one`, mientras que el resto son heredados de `QDA` (y transitivamente de `BaseBayesianClassifier`).

Los pasos que realiza `TensorizedQDA` para predecir lo mismo que `QDA` son los siguientes:

1. Ajuste (método `fit`, heredado de `BaseBayesianClassifier`)
    1. Se calculan las probabilidades a priori de pertenecer a alguna de las clases
        1. Esto se realiza mediante el método `_estimate_a_priori`, ambos métodos heredados de `BaseBayesianClassifier`
    1. Se calculan los siguientes valores en el método `_fit_params`:
        1. $\bar{x}_j$ o promedio de los $x$ de la clase *j*
        1. $s^2_j$ o matriz de covarianzas estimada para cada clase *j*
    1. Se tensorizan los valores calculados en el paso anterior mediante el método [np.stack](https://numpy.org/doc/stable/reference/generated/numpy.stack.html) de numpy, también en el método `_fit_params`
        1. Esto nos permitirá paralelizar la predicción de cada observación sobre las $k$ clases
1. Predicción (método `predict` heredado de `BaseBayesianClassifier`)
    1. Se realiza la predicción observación a observación (es decir, NO paralelizada), mediante el método `_predict_one` y `_predict_log_conditional`
    1. En el método `_predict_log_conditional`:
        1. Se realiza el cálculo del segundo término de $\log{f_j(x)} = -\frac{1}{2}\log |\Sigma_j| - \frac{1}{2} (x-\mu_j)^T \Sigma_j^{-1} (x- \mu_j) + C$
        1. Para ello, se calcula primera la diferencia de cada observación y la tensorización de las medias ($x-\mu_j$, siendo $\mu_j$ el atributo `tensor_means`), guardando su valor en la variable `unbiased_x`.
        1. Posteriormente, se calcula el valor del producto correspondiente a la fórmula $(x-\mu_j)^T \Sigma_j^{-1} (x- \mu_j)$ y se guarda su resultado en la variable `inner_prod`, paralelizando sobre las clases.
            1. Cabe resaltar que en este producto se usa el valor de `unbiased_x` transpuesto, el valor de `tensor_inv_cov`, y el valor de `unbiased_x`


### Código para la resolución del Punto 1

In [102]:
# levantamos el dataset Wine, que tiene 13 features y 178 observaciones en total
X_full, y_full = get_wine_dataset()

# encodeamos a número las clases
y_full_encoded = label_encode(y_full)

clases = np.unique(y_full_encoded)
print(f"Cantidad de clases: {len(clases)}\nClases: {clases}")

Cantidad de clases: 3
Clases: [0 1 2]


In [108]:
# Creamos un benchmark simple (similar a los dados como ejemplo) para facilitar el análisis de 
# 'tensor_inv_cov' y 'tensor_means' de TensorizedQDA
benchmark_pto_01 = Benchmark(
    X_full, y_full_encoded,
    n_runs = 100,
    warmup = 20,
    mem_runs = 20,
    test_sz = 0.3,
    same_splits = False
)

# Hacemos benchmark de QDA y TensorizedQDA
to_bench = [QDA, TensorizedQDA]
models = []
models_data = []

# Realizamos los benchmarks, y mostramos las properties que nos interesan
for model in to_bench:
    processed_model = benchmark_pto_01.bench(model)
    models.append(processed_model)
    models_data.append({
        "name": model,
        "inv_covs_type": type(processed_model.inv_covs),
        "inv_covs_len": len(processed_model.inv_covs),
        "means_type": type(processed_model.means),
        "means_len": len(processed_model.means),
        "tensor_inv_cov_type": type(processed_model.tensor_inv_cov) if hasattr(processed_model, 'tensor_inv_cov') else None,
        "tensor_inv_cov_shape": processed_model.tensor_inv_cov.shape if hasattr(processed_model, 'tensor_inv_cov') else None,
        "tensor_means_type": type(processed_model.tensor_means) if hasattr(processed_model, 'tensor_means') else None,
        "tensor_means_shape": processed_model.tensor_means.shape if hasattr(processed_model, 'tensor_means') else None,
    })


Benching params:
Total runs: 140
Warmup runs: 20
Peak Memory usage runs: 20
Running time runs: 100
Train size rows (approx): 125
Test size rows (approx): 53
Test size fraction: 0.3


QDA (MEM):   0%|          | 0/20 [00:00<?, ?it/s]

QDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

TensorizedQDA (MEM):   0%|          | 0/20 [00:00<?, ?it/s]

TensorizedQDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

In [110]:
# Imprimimos los datos obtenidos en el paso anterior
for model_data in models_data:
    for attr in model_data:
        print(f"{attr}: {model_data[attr]}")
    print()

name: <class '__main__.QDA'>
inv_covs_type: <class 'list'>
inv_covs_len: 3
means_type: <class 'list'>
means_len: 3
tensor_inv_cov_type: None
tensor_inv_cov_shape: None
tensor_means_type: None
tensor_means_shape: None

name: <class '__main__.TensorizedQDA'>
inv_covs_type: <class 'list'>
inv_covs_len: 3
means_type: <class 'list'>
means_len: 3
tensor_inv_cov_type: <class 'numpy.ndarray'>
tensor_inv_cov_shape: (3, 13, 13)
tensor_means_type: <class 'numpy.ndarray'>
tensor_means_shape: (3, 13, 1)



## Punto 2 - Optimización

Debido a la forma cuadrática de QDA, no se puede predecir para $n$ observaciones en una sola pasada (utilizar $X \in \mathbb{R}^{p \times n}$ en vez de $x \in \mathbb{R}^p$) sin pasar por una matriz de $n \times n$ en donde se computan todas las interacciones entre observaciones. Se puede acceder al resultado recuperando sólo la diagonal de dicha matriz, pero resulta ineficiente en tiempo y (especialmente) en memoria. Aún así, es *posible* que el modelo funcione más rápido.

3. Implementar el modelo `FasterQDA` (se recomienda heredarlo de `TensorizedQDA`) de manera de eliminar el ciclo for en el método predict.
4. Mostrar dónde aparece la mencionada matriz de $n \times n$, donde $n$ es la cantidad de observaciones a predecir.
5. Demostrar que
$$
diag(A \cdot B) = \sum_{cols} A \odot B^T = np.sum(A \odot B^T, axis=1)
$$ es decir, que se puede "esquivar" la matriz de $n \times n$ usando matrices de $n \times p$. También se puede usar, de forma equivalente,
$$
np.sum(A^T \odot B, axis=0).T
$$
queda a preferencia del alumno cuál usar.

6. Utilizar la propiedad antes demostrada para reimplementar la predicción del modelo `FasterQDA` de forma eficiente en un nuevo modelo `EfficientQDA`.
7. Comparar la performance de las 4 variantes de QDA implementadas hasta ahora (no Cholesky) ¿Qué se observa? A modo de opinión ¿Se condice con lo esperado?

### Respuestas Punto 2

#### Respuesta Punto 2.3 - Implementar el modelo `FasterQDA`...

Implementar el modelo `FasterQDA` (se recomienda heredarlo de `TensorizedQDA`) de manera de eliminar el ciclo for en el método predict.

##### Código para resolver el Punto 2.3

In [550]:
class FasterQDA(TensorizedQDA):
    def _fit_params(self, X, y):
        super()._fit_params(X,y)

    def _predict_log_conditionals(self,X):
        unbiased_x = X - self.tensor_means

        inner_prod = unbiased_x.transpose(0,2,1) @ self.tensor_inv_cov @ unbiased_x

        return (0.5*np.log(LA.det(self.tensor_inv_cov))).reshape(-1,1) - 0.5 * inner_prod.diagonal(axis1=1, axis2=2)
    
    def predict(self, X):
        return np.argmax(self.log_a_priori.reshape(-1,1) + self._predict_log_conditionals(X), axis=0)

In [551]:
# Hacemos un benchmark para evaluar la nueva implementación
to_bench_2_3 = [FasterQDA]

bench_2_3_wines = Benchmark(
    X_full_wine, y_full_encoded_wine,
    n_runs = 100,
    warmup = 20,
    mem_runs = 20,
    test_sz = 0.3,
    same_splits = False
)

for model in to_bench_2_3:
    bench_2_3_wines.bench(model)

bench_2_3_wines_summ = bench_2_3_wines.summary()[['train_median_ms', 'test_median_ms','mean_accuracy']]

display(bench_2_3_wines_summ)

Benching params:
Total runs: 140
Warmup runs: 20
Peak Memory usage runs: 20
Running time runs: 100
Train size rows (approx): 125
Test size rows (approx): 53
Test size fraction: 0.3


FasterQDA (MEM):   0%|          | 0/20 [00:00<?, ?it/s]

FasterQDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

,train_median_ms,test_median_ms,mean_accuracy
model,,,
FasterQDA,1.3473,0.333,0.982407


#### Respuesta Punto 2.4 - Mostrar dónde aparece la mencionada matriz de $n \times n$, donde $n$ es la cantidad de observaciones a predecir.

Estos son los pasos que se realizaron para implementar `FasterQDA` (ver implementación en `Respuesta Punto 2.3`):

1. Heredamos la implementación de `TensorizedQDA`
1. Modificamos el método `predict`, para que directamente llame al método `_predict_log_conditionals`
1. Modificamos el método `_predict_log_conditionals` de la siguiente manera:
    1. En vez de pasar el vector `x` ($x \in \mathbb{R}^p$), le pasamos la matriz `X` ($X \in \mathbb{R}^{p \times n}$)
    1. Esto modifica las operaciones de las siguientes variables:
        1. `unbiased_x`: ya que ahora se operan la matriz `X`, con $X \in \mathbb{R}^{p \times n}$, y `self.tensor_means`, con `tensor_means` $\in \mathbb{R}^{k \times p \times 1}$. Esto da como resultado un tensor $\in \mathbb{R}^{k \times p \times n}$
        1. `inner_prod`: ya que intervienen el tensor `tensor_means` traspuesto, con `tensor_means`$^T$ $\in \mathbb{R}^{k \times n \times p}$, el tensor `tensor_inv_cov` $\in \mathbb{R}^{k \times p \times p}$, y `tensor_means` $\in \mathbb{R}^{k \times p \times n}$
            1. > Como resultado, tenemos a la matriz $nxn$ mencionada en el enunciado (técnicamente, un tensor $\in \mathbb{R}^{k \times n \times n}$)

Recordemos que: 

* $k$ representa la cantidad de clases
* $n$ representa la cantidad de observaciones
* $p$ representa la cantidad de features/variables/predictores

#### Respuesta Punto 2.5 - Demostrar que...

##### Enunciado Punto 2.5

Demostrar que
$$
diag(A \cdot B) = \sum_{cols} A \odot B^T = np.sum(A \odot B^T, axis=1)
$$ es decir, que se puede "esquivar" la matriz de $n \times n$ usando matrices de $n \times p$. También se puede usar, de forma equivalente,
$$
np.sum(A^T \odot B, axis=0).T
$$
queda a preferencia del alumno cuál usar.

##### Demostración Punto 2.5

Primero que nada, definamos las dimensiones de las matrices $A$ y $B$.

Como podemos notar, en la igualdad mencionada en el enunciado ($diag(A \cdot B) = \sum_{cols} A \odot B^T$) tenemos que realizar dos operaciones con las matrices:
- Producto entre $A$ y $B$: lo cual indica que $A \in \mathbb{R}^{a \times N}$, y $B \in \mathbb{R}^{N \times b}$
- Producto elemento a elemento entre $A$ y $B^{T}$ (es decir, $A \odot B^T$): estos solo es posible si ambos tienen la mismas dimensiones 

De lo anterior, podemos decir que la única situación donde se cumplen es cuando tanto $A$ y $B$ son cuadradas y tienen la mismas dimenciones, es decir $A, B \in \mathbb{R}^{N \times N}$

Supongamos las siguientes matrices:

$
A =
  \begin{bmatrix}
    a_{1,1} & a_{1,2} & a_{1,3} & ... & a_{1,n} \\
    a_{2,1} & a_{2,2} & a_{2,3} & ... & a_{2,n} \\
    a_{3,1} & a_{3,2} & a_{3,3} & ... & a_{3,n} \\
    ... & ... & ... & ... \\
    a_{n,1} & a_{n,2} & a_{n,3} & ... & a_{n,n}
  \end{bmatrix}
$

$
B =
  \begin{bmatrix}
    b_{1,1} & b_{1,2} & b_{1,3} & ... & b_{1,n} \\
    b_{2,1} & b_{2,2} & b_{2,3} & ... & b_{2,n} \\
    b_{3,1} & b_{3,2} & b_{3,3} & ... & b_{3,n} \\
    ... & ... & ... & ... \\
    b_{n,1} & b_{n,2} & b_{n,3} & ... & b_{n,n}
  \end{bmatrix}
$

$
B^{T} =
  \begin{bmatrix}
    b_{1,1} & b_{2,1} & b_{3,1} & ... & b_{n,1} \\
    b_{1,2} & b_{2,2} & b_{3,2} & ... & b_{n,2} \\
    b_{1,3} & b_{2,3} & b_{3,3} & ... & b_{n,3} \\
    ... & ... & ... & ... \\
    b_{1,n} & b_{2,n} & b_{3,n} & ... & b_{n,n}
  \end{bmatrix}
$

Comencemos con la expresión $diag(A \cdot B)$. Como solo nos interesa la diagonal, en la siguiente expresión obviaremos a aquellos elementos que no estén en la misma (para simplificar las cuentas):

$
A \cdot B =
  \begin{bmatrix}
    \sum_{i=1}^{n} a_{1,i} * b_{i,1} & . & . & ... & . \\
    . & \sum_{i=1}^{n} a_{2,i} * b_{i,2} & . & ... & . \\
    . & . & \sum_{i=1}^{n} a_{3,i} * b_{i,3} & ... & . \\
    . & . & . & ... & \sum_{i=1}^{n} a_{n,i} * b_{i,n}
  \end{bmatrix}
$

Por lo cual podemos expresar lo siguiente:

$diag(A \cdot B) = [\sum_{i=1}^{n} a_{1,i} * b_{i,1}, \sum_{i=1}^{n} a_{2,i} * b_{i,2}, \sum_{i=1}^{n} a_{3,i} * b_{i,3}, ...,  \sum_{i=1}^{n} a_{n,i} * b_{i,n}]$

Ahora, sigamos con la operación $\sum_{cols} A \odot B^T$:

$
A \odot B^{T} =
  \begin{bmatrix}
    a_{1,1} * b_{1,1} & a_{1,2} * b_{2,1} & a_{1,3} * b_{3,1} & ... & a_{1,n} * b_{n,1} \\
    a_{2,1} * b_{1,2} & a_{2,2} * b_{2,2} & a_{2,3} * b_{3,2} & ... & a_{2,n} * b_{n,2} \\
    a_{3,1} * b_{1,3} & a_{3,2} * b_{2,3} & a_{3,3} * b_{3,3} & ... & a_{3,n} * b_{n,3} \\
    ... & ... & ... & ... \\
    a_{n,1} * b_{1,n} & a_{n,2} * b_{2,n} & a_{n,3} * b_{3,n} & ... & a_{n,n} * b_{n,n}
  \end{bmatrix}
$

El siguiente paso, consta de sumar columna por columna, tal como lo indica $\sum_{cols} A \odot B^T$, por lo cual, queda el siguiente vector:

$\sum_{cols} A \odot B^T = [\sum_{i=1}^{n} a_{1,i} * b_{i,1}, \sum_{i=1}^{n} a_{2,i} * b_{i,2}, \sum_{i=1}^{n} a_{3,i} * b_{i,3}, ...,  \sum_{i=1}^{n} a_{n,i} * b_{i,n}] $

Es decir, demostramos que $diag(A \cdot B) = \sum_{cols} A \odot B^T$.

Por último, podemos notar que en la expresión $\sum_{cols} A \odot B^T$ lo que se hace es:
1. $A \odot B^T$
1. sumar columna a columna

Esto en python se expresa como $np.sum(A \odot B^T, axis=1)$. Nótese que [numpy.sum](https://numpy.org/doc/2.1/reference/generated/numpy.sum.html) con `axis=1` indica sumar columna a columna.

Finalmente, podemos concluir que la expresión dada en el enunciado es verdadera: 

$diag(A \cdot B) = \sum_{cols} A \odot B^T = np.sum(A \odot B^T, axis=1)$

#### Respuesta Punto 2.6 - Utilizar la propiedad antes demostrada para reimplementar la predicción del modelo `FasterQDA` de forma eficiente en un nuevo modelo `EfficientQDA`.

En este caso, lo que debemos hacer es multiplicar dos matrices $A, B, \in \mathbb{R}^{n \times p}$ elemento a elemento, y luego utilizar el método $np.sum(A \cdot b, axis=1)$

La matriz $A$ será para nosotros el producto matricial de $X-\bar{x}$ (variable `unbiased_x`) traspuesto ($\in \mathbb{R}^{3 \times n \times p}$) con la matriz inversa de covarianzas ($\in \mathbb{R}^{3 \times p \times p}$). Nótese que al trasponer el primer elemento de este producto utilizamos `unbiased_x.transpose(0,2,1)`, ya que se trata de un tensor.

La matriz B constará también de $X-\bar{x}$ (variable `unbiased_x`) traspuesto ($\in \mathbb{R}^{3 \times n \times p}$), siguiendo el mismo método para trasponerla.

Finalmente, tendremos que aplicar la fórmula $np.sum(A \cdot b, axis=2)$. Nótose que utilizamos el `axis=2` ya que debemos utilizar las columnas de las matrices (término correspondiente a $p$), pero nuestra variable es un tensor de $k \times n \times p$.

##### Código para resolver el Punto 2.6

In [604]:
class EfficientQDA(TensorizedQDA):
    def _fit_params(self, X, y):
        super()._fit_params(X,y)

    def _predict_log_conditionals(self,X):
        unbiased_x = X - self.tensor_means

        n_p_matrix = unbiased_x.transpose(0,2,1) @ self.tensor_inv_cov
        inner_prod = np.sum(n_p_matrix * unbiased_x.transpose(0,2,1), axis=2)

        return (0.5*np.log(LA.det(self.tensor_inv_cov))).reshape(-1,1) - 0.5 * inner_prod
    
    def predict(self, X):
        return np.argmax(self.log_a_priori.reshape(-1,1) + self._predict_log_conditionals(X), axis=0)

In [603]:
# Hacemos un benchmark para evaluar la nueva implementación
to_bench_2_6 = [EfficientQDA]

bench_2_6_wines = Benchmark(
    X_full_wine, y_full_encoded_wine,
    n_runs = 100,
    warmup = 20,
    mem_runs = 20,
    test_sz = 0.3,
    same_splits = False
)

for model in to_bench_2_6:
    bench_2_6_wines.bench(model)

bench_2_6_wines_summ = bench_2_6_wines.summary()[['train_median_ms', 'test_median_ms','mean_accuracy']]

display(bench_2_6_wines_summ)

Benching params:
Total runs: 140
Warmup runs: 20
Peak Memory usage runs: 20
Running time runs: 100
Train size rows (approx): 125
Test size rows (approx): 53
Test size fraction: 0.3


EfficientQDA (MEM):   0%|          | 0/20 [00:00<?, ?it/s]

EfficientQDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

,train_median_ms,test_median_ms,mean_accuracy
model,,,
EfficientQDA,1.2244,0.24635,0.982407


#### Respuesta Punto 2.7 - Comparar la performance de las 4 variantes de QDA implementadas hasta ahora (no Cholesky) ¿Qué se observa? A modo de opinión ¿Se condice con lo esperado?

En la siguiente tabla se puede observar la comparación de performance de `QDA`, `TensorizedQDA`, `FasterQDA` y `EfficientQDA`, considerando la siguiente configuración de benchmark:

- *Datasets Utilizados*:
  - Wine dataset (178 observaciones, 13 features)
  - Iris dataset (150 observaciones, 4 features)
  - Penguins dataset (342 observaciones, 4 features)
- *N runs*: 100
- *Warmup*: 20
- *Mem runs*: 20
- *Test size*: 30%

| | `Wines` - train median ms | `Wines` - test median ms | `Wines` - mean accuracy | `Iris` - train median ms | `Iris` - test median ms | `Iris` - mean accuracy | `Penguins` - train median ms | `Penguins` - test median ms | `Penguins` - mean accuracy |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| QDA | 1.04780 | 8.21550 | 0.982407 | 0.69360 | 5.22535 | 0.970667 | 0.87535 | 13.22025 | 0.986408 |
| TensorizedQDA | 1.24735 | 3.82300 | 0.982593 | 1.34090 | 3.74830 | 0.971111 | 1.89745 | 12.01950 | 0.987379 |
| FasterQDA | 4.02435 | 1.25945 | 0.985741 | 5.29315 | 0.76765 | 0.972889 | 3.20085 | 0.87540 | 0.986699 |
| EfficientQDA | 2.67610 | 0.49290 | 0.983333 | 2.57925 | 0.40135 | 0.972222 | 2.61350 | 0.43440 | 0.988252 |

Como se puede observar, `FasterQDA` y `EfficientQDA` tienen menores tiempos (medios) en ***test*** en comparación con `QDA` y `TensorizedQDA` en todos los datasets, siendo `Penguins` datasets donde la diferencia es mayor: menos de *1 ms* para `FasterQDA` y `EfficientQDA`, y más de *12 ms* para `QDA` y `TensorizedQDA`. En cambio, al comparar los tiempos (medios) en ***train***, notamos `FasterQDA` y `EfficientQDA` demoran más que `QDA` y `TensorizedQDA`.

Respecto al mean ***accuracy***, `FasterQDA` y `EfficientQDA` superan a `QDA` y `TensorizedQDA` en todos los datasets, salvo para el caso de `Penguins`, donde el mean accuracy de `TensorizedQDA` es mayor al de `FasterQDA`, pero menor al de `EfficientQDA`.

Esto se condice con lo esperado, es decir, que las implementaciones de `FasterQDA` y `EfficientQDA` sean mejores que las de `QDA` y `TensorizedQDA` no solo en accuracy, sino también en tiempos medios (por lo menos para *test*).

##### Código para la resolución del Punto 2.7

In [607]:
# Definimos las variantes de QDA sobre las que vamos a hacer benchmarking
to_bench_2_7 = [QDA, TensorizedQDA, FasterQDA, EfficientQDA]

# Definimos un benchmark por cada dataset

bench_2_7_wines = Benchmark(
    X_full_wine, y_full_encoded_wine,
    n_runs = 100,
    warmup = 20,
    mem_runs = 20,
    test_sz = 0.3,
    same_splits = False
)

bench_2_7_iris = Benchmark(
    X_full_iris, y_full_encoded_iris,
    n_runs = 100,
    warmup = 20,
    mem_runs = 20,
    test_sz = 0.3,
    same_splits = False
)

bench_2_7_penguins = Benchmark(
    X_full_penguins, y_full_encoded_penguins,
    n_runs = 100,
    warmup = 20,
    mem_runs = 20,
    test_sz = 0.3,
    same_splits = False
)

Benching params:
Total runs: 140
Warmup runs: 20
Peak Memory usage runs: 20
Running time runs: 100
Train size rows (approx): 125
Test size rows (approx): 53
Test size fraction: 0.3
Benching params:
Total runs: 140
Warmup runs: 20
Peak Memory usage runs: 20
Running time runs: 100
Train size rows (approx): 105
Test size rows (approx): 45
Test size fraction: 0.3
Benching params:
Total runs: 140
Warmup runs: 20
Peak Memory usage runs: 20
Running time runs: 100
Train size rows (approx): 240
Test size rows (approx): 102
Test size fraction: 0.3


In [608]:
for model in to_bench_2_7:
    bench_2_7_wines.bench(model)
    bench_2_7_iris.bench(model)
    bench_2_7_penguins.bench(model)

QDA (MEM):   0%|          | 0/20 [00:00<?, ?it/s]

QDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

QDA (MEM):   0%|          | 0/20 [00:00<?, ?it/s]

QDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

QDA (MEM):   0%|          | 0/20 [00:00<?, ?it/s]

...

EfficientQDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

EfficientQDA (MEM):   0%|          | 0/20 [00:00<?, ?it/s]

EfficientQDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

EfficientQDA (MEM):   0%|          | 0/20 [00:00<?, ?it/s]

EfficientQDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

In [609]:
bench_2_7_wines_summ = bench_2_7_wines.summary()[['train_median_ms', 'test_median_ms','mean_accuracy']]
bench_2_7_iris_summ = bench_2_7_iris.summary()[['train_median_ms', 'test_median_ms','mean_accuracy']]
bench_2_7_penguins_summ = bench_2_7_penguins.summary()[['train_median_ms', 'test_median_ms','mean_accuracy']]

bench_2_7_wines_summ.columns = "Wines__" + bench_2_7_wines_summ.columns
bench_2_7_iris_summ.columns = "Iris__" + bench_2_7_iris_summ.columns
bench_2_7_penguins_summ.columns = "Penguins__" + bench_2_7_penguins_summ.columns

bench_2_7_comparisson = pd.concat([bench_2_7_wines_summ, bench_2_7_iris_summ, bench_2_7_penguins_summ], ignore_index=False, axis=1)

display(bench_2_7_comparisson)

,Wines__train_median_ms,Wines__test_median_ms,Wines__mean_accuracy,Iris__train_median_ms,Iris__test_median_ms,Iris__mean_accuracy,Penguins__train_median_ms,Penguins__test_median_ms,Penguins__mean_accuracy
model,,,,,,,,,
QDA,1.04780,8.21550,0.982407,0.69360,5.22535,0.970667,0.87535,13.22025,0.986408
TensorizedQDA,1.24735,3.82300,0.982593,1.34090,3.74830,0.971111,1.89745,12.01950,0.987379
FasterQDA,4.02435,1.25945,0.985741,5.29315,0.76765,0.972889,3.20085,0.87540,0.986699
EfficientQDA,2.67610,0.49290,0.983333,2.57925,0.40135,0.972222,2.61350,0.43440,0.988252


## Punto 3 - Cholesky - Diferencias entre implementaciones de `QDA_Chol`

### Cholesky

Hasta ahora todos los esfuerzos fueron enfocados en realizar una predicción más rápida. Los tiempos de entrenamiento (teóricos al menos) siguen siendo los mismos o hasta (minúsculamente) peores, dado que todas las mejoras siguen llamando al método `_fit_params` original de `QDA`.

La descomposición/factorización de [Cholesky](https://en.wikipedia.org/wiki/Cholesky_decomposition#Statement) permite factorizar una matriz definida positiva $A = LL^T$ donde $L$ es una matriz triangular inferior. En particular, si bien se asume que $p \ll n$, invertir la matriz de covarianzas $\Sigma$ para cada clase impone un cuello de botella que podría alivianarse. Teniendo en cuenta que las matrices de covarianza son simétricas y salvo degeneración, definidas positivas, Cholesky como mínimo debería permitir invertir la matriz más rápido.

*Nota: observar que calcular* $A^{-1}b$ *equivale a resolver el sistema* $Ax=b$.

### 3) Diferencias entre implementaciones de `QDA_Chol`

8. Si una matriz $A$ tiene fact. de Cholesky $A=LL^T$, expresar $A^{-1}$ en términos de $L$. ¿Cómo podría esto ser útil en la forma cuadrática de QDA?
7. Explicar las diferencias entre `QDA_Chol1`y `QDA` y cómo `QDA_Chol1` llega, paso a paso, hasta las predicciones.
8. ¿Cuáles son las diferencias entre `QDA_Chol1`, `QDA_Chol2` y `QDA_Chol3`?
9. Comparar la performance de las 7 variantes de QDA implementadas hasta ahora ¿Qué se observa?¿Hay alguna de las implementaciones de `QDA_Chol` que sea claramente mejor que las demás?¿Alguna que sea peor?

### Respuestas Punto 3

#### Respuesta Punto 3.8 - Si una matriz $A$ tiene fact. de Cholesky...

> Si una matriz $A$ tiene fact. de Cholesky $A=LL^T$, expresar $A^{-1}$ en términos de $L$. ¿Cómo podría esto ser útil en la forma cuadrática de QDA?

1. Partiendo de la Definición de factorización de Cholesky
    - $A=LL^T$
1. Invertimos a ambos lados de la ecuación, considerando que la inversa del productio de matrices $AB$ es igual al productio de sus inversas en orden inverso, es decir $B^{-1}A^{-1}$
    - $A^{-1}=(L^T)^{-1}L^{-1}$
1. Considerando que la inversa de la transpuesta de una matriz es igual a la transpuesta de la matriz inversa, escribimos:
    - $A^{-1}=(L^{-1})^{T}L^{-1}$
1. Es decir, dada una matriz $A$ que tienen factorización de Cholesky, podemos expresar a su inversa $A{-1}$ como $(L^{-1})^{T}L^{-1}$, siendo $L$ una *matriz triangular inferior*.
1. Ahora bien, recordando que queremos resolver el sistema $Ax=b$, podemos valernos de $A^{-1}b$ junto a la igualdad obtenida anteriormente, quedando:
    - $(L^{-1})^{T}L^{-1}b$

En la forma cuadrática de QDA podemos utilizar esta propiedad para calcular la inversa de la matriz de covarianzas $\Sigma$ de una forma, en principio, más eficiente en cuanto al consumo de procesador y memoria,
ya que solamente deberíamos calcular, en principio, la matriz $L$, luego invertirla, y luego multiplicar su transpuesta por dicha inversa.

> ¿Cómo podría esto ser útil en la forma cuadrática de QDA?

Recordemos que en QDA resolvemos buscamos resolver esta fórmula:

$$
\log{f_j(x)} = -\frac{1}{2}\log |\Sigma_j| - \frac{1}{2} (x-\mu_j)^T \Sigma_j^{-1} (x- \mu_j) + C
$$

1. El primer uso que podemos darle a esta propiedad que obtuvimos es en el primer término, al calcular el determinante de la matriz inversa de covarianzas, o $|\Sigma_j|$, ya que el determinante de una matriz triangular es el producto de los elementos de la diagonal.
    1. En nuestra fórmula, esto queda expresado como $-\frac{1}{2}\log(diag((L^{-1})^{T})*diag(L^{-1}))$
    1. Considerando que $diag((L^{-1})^{T}) = diag(L^{-1})$, expresamos lo anterior como $-\frac{1}{2}\log(diag(L^{-1})*diag(L^{-1})) = -\frac{1}{2}\log(diag(L^{-1})^{2})$
    1. Aprovechando las propiedades de los logaritmos, nos queda $\frac{1}{2}*2\log(diag(L^{-1}))=\log(diag(L^{-1}))$
    1. Es decir, reducimos la expresión del primer término a:
        1.  $\log(diag(L^{-1}))$
1. El segundo uso lo encontramos en el segundo término, en donde se hace $(x-\mu_j)^T \Sigma_j^{-1} (x- \mu_j)$, en donde también podemos usar la propiedad anterior de la siguiente manera:
    1. Consideremos:
        1. $(x-\mu_j)$ como $b$
        1. $\Sigma_j^{-1}$ como $A^{-1}$
    1. Reemplazando, queda $b^T*A^{-1}*b$
    1. Utilizando la propiedad de factorización de Cholesky, obtenemos:
        1. $b^T*(L^{-1})^{T}L^{-1}*b$
    1. Valiéndonos de la propiedad de transposición de Matrices y Vectores, es decir, $(Ax)^T = x^TA^T$, podemos expresar lo siguiente:
        1. $[b^T*(L^{-1})^{T}][L^{-1}*b]  = [L^{-1}*b] * [b^T*(L^{-1})^{T}]^T = [L^{-1}*b] *[L^{-1}*b] = [L^{-1}*b]^2 $
    1. Por lo tanto, el segundo término nos queda:
        1. $-\frac{1}{2}(L^{-1}*b)^2$

En conclusión, podemos valernos de la factorizaci;on de Cholesky en QDA, ya que nos facilita el cálculo de $\log{f_j(x)}$ de la siguiente manera:

- el primer término se se puede calcular como el logaritmo del producto de los elementos de $L^{-1}$
- el segundo término se reduce al producto de $-\frac{1}{2}$ con $(L^{-1}*b)^2$, siendo $b = (x-\mu_j)$

> NOTA 1: Una matriz triangular es invertible si y solo si todos los elementos de la diagonal son no nulos. En este caso, la inversa de una matriz triangular inferior es otra matriz triangular inferior. [Fuente](https://es.wikipedia.org/wiki/Matriz_triangular#Propiedades_de_las_matrices_triangulares).

> NOTA 2: El determinante de una matriz triangular es el producto de los elementos de la diagonal. [Fuente](https://es.wikipedia.org/wiki/Matriz_triangular#Propiedades_de_las_matrices_triangulares).

#### Respuesta Punto 3.9 - Explicar las diferencias entre `QDA_Chol1`y `QDA` y cómo `QDA_Chol1` llega, paso a paso, hasta las predicciones.

Las diferencias entre `QDA` y `QDA_Chol1` se encuentran en las implementaciones de los siguientes métodos (Nota: el resto de los métodos son heredados de `BaseBayesianClassifier`):
- `_fit_params`
    - `QDA` calcula y crea los atributes `inv_covs` y `means`.
        - `inv_covs` es la inversa de la matriz de covarianzas $\Sigma_j$ para la clase `j`
        - `means` consta de la media $\bar{x}_j$, utilizada para estimar $\mu_j$ para la clase `j`
    - `QDA_Chol1` calcula y crea los atributes `L_invs` y `means`.
        - `L_invs` consta de la matriz inversa de L, o $L^{-1}$, es decir, se aprovecha de la propiedad descrita en el ***punto 3.8*** para facilitar el cálculo de la inversa $\Sigma_j$.
        - `means` consta de la media $\bar{x}_j$, utilizada para estimar $\mu_j$ para la clase `j`. Se calcula de la misma manera que en `QDA`
- `_predict_log_conditional`
    - `QDA` calcula el $log(P(x|G=class_idx))$ mediante la fórmula $\log{f_j(x)} = -\frac{1}{2}\log |\Sigma_j| - \frac{1}{2} (x-\mu_j)^T \Sigma_j^{-1} (x- \mu_j) + C$, utilizando los estimadores ($\Sigma_j^{-1}$ y $\mu_j$) del método `_fit_params`
    - `QDA_Chol1` a diferencia del método implementado por `QDA`, aquí se resuelve el valor de $log(P(x|G=class_idx))$ a partir del valor de `L_invs` o $L^{-1}$. `NOTA`: ver detalles de como se implementa $L^{-1}$ en QDA descritos en el `Punto 3.8`.

- ¿Cómo `QDA_Chol1` llega hasta las predicciones?
    1. Como se mencionó anteriormente, `QDA_Chol1` obtiene estos valores en el método `_fit_params`:
        1. Estima el valor de $\mu_j$ mediante el $\bar{x}_j$, y lo guarda en el attribute `means`
        1. Calcula la inversa de matriz de covarianas $\Sigma_j$, o $L^{-1}$, y guarda el valor en `L_invs`
    1. Luego, en el método `predict` (heredado desde `BaseBayesianClassifier`), llama al método `_predict_one`, el cual, para cada observación $n$ (representada por el vector `x`) y por cada clase $k$ (representada por `idx`, llama al método `_predict_log_conditional`
    1. En el método `_predict_log_conditional` es donde se obtienen las predicciones, y se hace de la siguiente manera:
        1. A partir del atributo `L_invs`, se obtiene la matriz $L^{-1}$ correspondiente a la clase $k$ sobre la cual se está trabajando
        1. Luego se calcula la diferencia entre la observación $n$ con el valor de medias `means`, es decir, se hace el cálculo correspondiente a $x-\mu_j$
        1. Se calcula el producto entre la matriz diagonal inversa $L^{-1}$ y el vector ($x-\mu_j$), y se guarda el valor en la variable $y$
        1. Se obtiene el logaritmo de la probabilidad condicional de que la observación $x$ pertenezca a la clase $k$ de la siguiente manera:
            1. Se calcula la diagonal de (L^{-1}), el cual se obtiene al multiplicar los valores de su diagonal principal (`NOTA`: ver detalles de como se llega a este término en el `Punto 3.8`)
            1. Se calcula el cuadrado de la variable $y$, y se multiplica por $-0.5$ (`NOTA`: ver detalles de como se llega a este término en el `Punto 3.8`)
            1. Se devuelve la suma de los valores anteriores

#### Respuesta Punto 3.10 - ¿Cuáles son las diferencias entre `QDA_Chol1`, `QDA_Chol2` y `QDA_Chol3`?

Primero que nada, remarquemos que tanto `QDA_Chol1`, `QDA_Chol2` y `QDA_Chol3` tienen en común:

- Las tres heredan de la clase `BaseBayesianClassifier`
- Las tres tienen sus propias implementaciones del método `_fit_params` y del método `_predict_log_conditional`
  - `QDA_Chol1` y `QDA_Chol3` tienen la misma implementación, mientras que `QDA_Chol2` tiene ciertas diferencias que explicaremos a continuación.

***Diferencias entre `QDA_Chol1` y `QDA_Chol2`***

- `_fit_params`
  - Como mencionamos en los puntos `3.8` y `3.9`, `QDA_Chol1` calcula y crea los attributes `L_invs` y `means`:
    - `L_invs` consta de la matriz inversa $L^{-1}$, es decir, se aprovecha de la propiedad descrita en el ***punto 3.8*** para facilitar el cálculo de la inversa $\Sigma_j$.
    - `means` consta de la media $\bar{x}_j$, utilizada para estimar $\mu_j$ para la clase `j`. Se calcula de la misma manera que en `QDA`
  - Por su parte, `QDA_Chol2` calcula y crea los attributes de `Ls` y `means`:
    - `means` representa exactamente lo mismo que en `QDA_Chol1`, es decir, es la media $\bar{x}_j$, utilizada para estimar $\mu_j$ para la clase `j`
    - `Ls` consta de la matriz $L$, es decir, a diferencia de `QDA_Chol1` no calcula su inversa.
        - Cabe destacar que el procedimiento para llegar a $L$ es el mismo en ambas clases, pero `QDA_Chol1` obtiene la inversa mediante el método [inv](https://docs.scipy.org/doc/scipy/reference/generated/scipy.linalg.inv.html) de la library `numpy.linalg` (alias `LA`).
- `_predict_log_conditional`
    - La primer diferencia se nota en el uso de matrices, siendo que `QDA_Chol1` usa $L^{-1}$, y `QDA_Chol2` usa $L$
    - Debido a esto, al calcular la variable `y`, que posteriormente será utilizada en el `return` de la función, tenemos estas diferencias:
      - `QDA_Chol1` hace uso de la fórmula $A^{-1}*b$, siendo $A^{-1} = L^{-1}$ y $b = $ unbiased_x $= x-\mu_j$
      - `QDA_Chol2` se caracteríza por utilizar la fórmula $Ax=b$, la cual resuelve mediante el método [solve_triangular](https://docs.scipy.org/doc/scipy/reference/generated/scipy.linalg.solve_triangular.html) de la library `numpy.linalg` (alias `LA`).

En conclusión, la diferencia principal entre `QDA_Chol1` y `QDA_Chol2` es que la primera hace uso de $L^{-1}$ junto con la fórmula $A^{-1}*b$ para resolver el logaritmo de las probabilidades a posteriori, mientras que `QDA_Chol2` utiliza la matriz `L` y la fórmula $Ax=b$ para lograr el mismo objetivo

***Diferencias entre `QDA_Chol1` y `QDA_Chol3`***

- `_fit_params`
  - La diferencia redica en como calculan la variable `L_invs` o matriz inversa $L^{-1}$
  - `QDA_Chol3`, a diferencia de `QDA_Chol1`, implementa el método [dtrtri](https://docs.scipy.org/doc/scipy/reference/generated/scipy.linalg.lapack.dtrtri.html) de la library `scipy.linalg.lapack` (técnicamente, es un wrapper de la función dtrtri del package [Lapack](https://www.netlib.org/lapack/explore-html/de/d61/group__trtri_ga2da4f285dfccde2ef75221144799a3ec.html)).
  - Esta función permite calcular la inversa de una matriz triangular superior o inferior.
  - Como en nuestro caso trabajamos con una matriz triangular inferior, debemos pasarle el parámetro `lower` en `True`

En conclusión, la diferencia principal entre `QDA_Chol1` y `QDA_Chol3` es que esta última hace uso de un package especializado para calcular la inversa de la matriz triangular inferior $L^{-1}$, en vez de calcularla mediante el método [inv](https://docs.scipy.org/doc/scipy/reference/generated/scipy.linalg.inv.html) del package `numpy.linalg` como hace la clase `QDA_Chol1`.

#### Respuesta Punto 3.11 - Comparar la performance de las 7 variantes de QDA implementadas hasta ahora ¿Qué se observa?¿Hay alguna de las implementaciones de `QDA_Chol` que sea claramente mejor que las demás?¿Alguna que sea peor?

En la siguiente tabla, se pueden observar la comparación de performance de los métodos `QDA`, `TensorizedQDA`, `FasterQDA`, `EfficientQDA`, `QDA_Chol1`, `QDA_Chol2`, `QDA_Chol3`, considerando la siguiente configuración de benchmark:

- *Datasets Utilizados*:
  - Wine dataset (178 observaciones, 13 features)
  - Iris dataset (150 observaciones, 4 features)
  - Penguins dataset (342 observaciones, 4 features)
- *N runs*: 100
- *Warmup*: 20
- *Mem runs*: 20
- *Test size*: 30%
- *Same splits*: False

| | `Wines` - train median ms | `Wines` - test median ms | `Wines` - mean accuracy | `Iris` - train_median_ms | `Iris` - test median ms | `Iris` - mean accuracy | `Penguins` - train median ms | `Penguins` - test median ms | `Penguins` - mean accuracy |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| QDA | 0.7246 | 5.93785 | 0.982407 | 1.10150 | 8.19635 | 0.970667 | 1.13870 | 18.08705 | 0.986408
| TensorizedQDA | 1.60685 | 5.57250 | 0.982593 | 1.72105 | 4.87040 | 0.971111 | 0.78600 | 4.54860 | 0.987379
| FasterQDA | 1.55645 | 0.40810 | 0.985741 | 0.97430 | 0.19800 | 0.972889 | 1.05215 | 0.25580 | 0.986699
| EfficientQDA | 1.30335 | 0.25285 | 0.983333 | 0.95270 | 0.16315 | 0.972222 | 0.76350 | 0.14665 | 0.988252
| QDA_Chol1 | 1.71515 | 9.24255 | 0.986111 | 1.85610 | 7.58600 | 0.977333 | 1.51250 | 14.04350 | 0.987282
| QDA_Chol2 | 1.38175 | 17.73100 | 0.982222 | 1.10275 | 12.28120 | 0.972667 | 1.39320 | 27.57175 | 0.986699
| QDA_Chol3 | 0.76960 | 4.28775 | 0.984444 | 0.76400 | 3.73935 | 0.972222 | 0.62925 | 6.71720 | 0.987282

Enfocándonos en las diferentes implementaciones de `QDA_Chol`, podemos observar que la implementación de `QDA_Chol3` tiene mejores métricas de tiempo medio de entrenamiento y test en los distintos datasets, a la vez que, en términos de `accuracy`, NO es el mejor para esos casos, pero tampoco presenta mucha diferencia con las otras dos implementaciones de `QDA_Chol`. Es más, la diferencia aparece recién en el tercer decimal.

Al comparar `QDA_Chol3` con `QDA` podemos notar que tiene mejores tiempos de train y test en todos los datasets, con excepción del tiempo medio de train en el dataset de *wines*, como así también mejor `accuracy` en todos los datasets.
Al comparar `QDA_Chol3` con `TensorizedQDA` notamos que el primero es mejor en todas las métricas en los datasets de *Wines* e *Iris*, mientras que en el dataset de *Penguins* mejora solamente los tiempos medios en `Train`. Respecto al accuracy en este último dataset, `TensorizedQDA` es mejor, pero la diferencia se nota recién en el cuarto decimal.
Al comparar `QDA_Chol3` con `FasterQDA` notamos que en general la implementación de `FasterQDA` es mejor en todas las métricas que `QDA_Chol3`, con la diferencia del dataset `Penguins`, en donde el accuracy para `QDA_Chol3` supera al de `FasterQDA`, pero la mejora se nota recién en el tercer decimal.
Por último, al comparar `QDA_Chol3` con `EfficientQDA` notamos que `EfficientQDA` tiene menores tiempos de *test* que `QDA_Chol3` en todos los datasets. Respecto al `accuracy`, `QDA_Chol3` solo lo supera en el dataset de *wines*.

En resúmen, la elección entre una de las implementaciones de `QDA_Chol` dependerá si buscamos el mejor tiempo (medio) de train y test, o el mejor `accuracy`:
- `Train/test median ms`: `QDA_Chol3`
- `Accuracy`: `QDA_Chol2`

En nuestra caso, procedemos a analizar en futuros puntos/ejercicios al `QDA_Chol3`, ya que tiene mejores tiempos de Train/test, y el `accuracy`, como mencionamos, difiere recién al tercer decimal.

##### Código para la resolución del ejercicio 3.11

In [610]:
# Definimos las variantes de QDA sobre las que vamos a hacer benchmarking
to_bench_3_11 = [QDA, TensorizedQDA, FasterQDA, EfficientQDA, QDA_Chol1, QDA_Chol2, QDA_Chol3]

bench_3_11_wines = Benchmark(
    X_full_wine, y_full_encoded_wine,
    n_runs = 100,
    warmup = 20,
    mem_runs = 20,
    test_sz = 0.3,
    same_splits = False
)

bench_3_11_iris = Benchmark(
    X_full_iris, y_full_encoded_iris,
    n_runs = 100,
    warmup = 20,
    mem_runs = 20,
    test_sz = 0.3,
    same_splits = False
)

bench_3_11_penguins = Benchmark(
    X_full_penguins, y_full_encoded_penguins,
    n_runs = 100,
    warmup = 20,
    mem_runs = 20,
    test_sz = 0.3,
    same_splits = False
)


Benching params:
Total runs: 140
Warmup runs: 20
Peak Memory usage runs: 20
Running time runs: 100
Train size rows (approx): 125
Test size rows (approx): 53
Test size fraction: 0.3
Benching params:
Total runs: 140
Warmup runs: 20
Peak Memory usage runs: 20
Running time runs: 100
Train size rows (approx): 105
Test size rows (approx): 45
Test size fraction: 0.3
Benching params:
Total runs: 140
Warmup runs: 20
Peak Memory usage runs: 20
Running time runs: 100
Train size rows (approx): 240
Test size rows (approx): 102
Test size fraction: 0.3


In [611]:
for model in to_bench_3_11:
    bench_3_11_wines.bench(model)
    bench_3_11_iris.bench(model)
    bench_3_11_penguins.bench(model)

QDA (MEM):   0%|          | 0/20 [00:00<?, ?it/s]

QDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

QDA (MEM):   0%|          | 0/20 [00:00<?, ?it/s]

QDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

QDA (MEM):   0%|          | 0/20 [00:00<?, ?it/s]

...

QDA_Chol3 (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

QDA_Chol3 (MEM):   0%|          | 0/20 [00:00<?, ?it/s]

QDA_Chol3 (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

QDA_Chol3 (MEM):   0%|          | 0/20 [00:00<?, ?it/s]

QDA_Chol3 (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

In [612]:
bench_3_11_wines_summ = bench_3_11_wines.summary()[['train_median_ms', 'test_median_ms','mean_accuracy']]
bench_3_11_iris_summ = bench_3_11_iris.summary()[['train_median_ms', 'test_median_ms','mean_accuracy']]
bench_3_11_penguins_summ = bench_3_11_penguins.summary()[['train_median_ms', 'test_median_ms','mean_accuracy']]

bench_3_11_wines_summ.columns = "Wines__" + bench_3_11_wines_summ.columns
bench_3_11_iris_summ.columns = "Iris__" + bench_3_11_iris_summ.columns
bench_3_11_penguins_summ.columns = "Penguins__" + bench_3_11_penguins_summ.columns

bench_3_11_comparisson = pd.concat([bench_3_11_wines_summ, bench_3_11_iris_summ, bench_3_11_penguins_summ], ignore_index=False, axis=1)

In [613]:
display(bench_3_11_comparisson)

,Wines__train_median_ms,Wines__test_median_ms,Wines__mean_accuracy,Iris__train_median_ms,Iris__test_median_ms,Iris__mean_accuracy,Penguins__train_median_ms,Penguins__test_median_ms,Penguins__mean_accuracy
model,,,,,,,,,
QDA,0.72460,5.93785,0.982407,1.10150,8.19635,0.970667,1.13870,18.08705,0.986408
TensorizedQDA,1.60685,5.57250,0.982593,1.72105,4.87040,0.971111,0.78600,4.54860,0.987379
FasterQDA,1.55645,0.40810,0.985741,0.97430,0.19800,0.972889,1.05215,0.25580,0.986699
EfficientQDA,1.30335,0.25285,0.983333,0.95270,0.16315,0.972222,0.76350,0.14665,0.988252
QDA_Chol1,1.71515,9.24255,0.986111,1.85610,7.58600,0.977333,1.51250,14.04350,0.987282
QDA_Chol2,1.38175,17.73100,0.982222,1.10275,12.28120,0.972667,1.39320,27.57175,0.986699
QDA_Chol3,0.76960,4.28775,0.984444,0.76400,3.73935,0.972222,0.62925,6.71720,0.987282


| | `Wines` - train median ms | `Wines` - test median ms | `Wines` - mean accuracy | `Iris` - train_median_ms | `Iris` - test median ms | `Iris` - mean accuracy | `Penguins` - train median ms | `Penguins` - test median ms | `Penguins` - mean accuracy |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| QDA | 0.7246 | 5.93785 | 0.982407 | 1.10150 | 8.19635 | 0.970667 | 1.13870 | 18.08705 | 0.986408
| TensorizedQDA | 1.60685 | 5.57250 | 0.982593 | 1.72105 | 4.87040 | 0.971111 | 0.78600 | 4.54860 | 0.987379
| FasterQDA | 1.55645 | 0.40810 | 0.985741 | 0.97430 | 0.19800 | 0.972889 | 1.05215 | 0.25580 | 0.986699
| EfficientQDA | 1.30335 | 0.25285 | 0.983333 | 0.95270 | 0.16315 | 0.972222 | 0.76350 | 0.14665 | 0.988252
| QDA_Chol1 | 1.71515 | 9.24255 | 0.986111 | 1.85610 | 7.58600 | 0.977333 | 1.51250 | 14.04350 | 0.987282
| QDA_Chol2 | 1.38175 | 17.73100 | 0.982222 | 1.10275 | 12.28120 | 0.972667 | 1.39320 | 27.57175 | 0.986699
| QDA_Chol3 | 0.76960 | 4.28775 | 0.984444 | 0.76400 | 3.73935 | 0.972222 | 0.62925 | 6.71720 | 0.987282


## Punto 4 - Optimización

***Optimización***

12. Implementar el modelo `TensorizedChol` paralelizando sobre clases/observaciones según corresponda. Se recomienda heredarlo de alguna de las implementaciones de `QDA_Chol`, aunque la elección de cuál de ellas queda a cargo del alumno según lo observado en los benchmarks de puntos anteriores.
13. Implementar el modelo `EfficientChol` combinando los insights de `EfficientQDA` y `TensorizedChol`. Si se desea, se puede implementar `FasterChol` como ayuda, pero no se contempla para el punto.
13. Comparar la performance de las 9 variantes de QDA implementadas ¿Qué se observa? A modo de opinión ¿Se condice con lo esperado?

### Respuestas Punto 4

#### Respuesta Punto 4.12 - Implementar el modelo `TensorizedChol` paralelizando sobre clases/observaciones según corresponda...

Implementar el modelo `TensorizedChol` paralelizando sobre clases/observaciones según corresponda. Se recomienda heredarlo de alguna de las implementaciones de `QDA_Chol`, aunque la elección de cuál de ellas queda a cargo del alumno según lo observado en los benchmarks de puntos anteriores.

##### Explicación del código para Punto 4.12 

Para esta implementación de `TensorizedChol` comenzamos heredando los métodos de `QDA_Chol3`, ya que en las pruebas realizadas en puntos anteriores notamos que, para determinados datasets, es el que menos tiempo medio demora en realizar los entrenamientos y tests, y si bien el accuracy no es el más alto, tampoco hay tanta diferencia entre los demás modelos.

La principal diferencia con `QDA_Chol3` es que tensorizamos los valores de $L^{-1}$ y $means$. Luego, en el método `_predict_log_conditionals` procedemos de manera similar a `QDA_Chol3`, salvo en el momento de realizar el return, donde debemos realizar las siguientes operaciones para garantizar que se devuelvan las estructuras de datos con las dimensiones correctas:

- Primero que nada, definimos las dimensiones de las variables con las cuales trabajamos
  - `self.tensor_L_inv` y `L_inv` (representan $L^{-1}$) tienen una dimensión de $(3, 13, 13)$, correspondiente a $k=3 clases, p=13 predictores, n=13 observaciones$.
  - `self.tensor_means` y `unbiased_x` (representan a $(x-\mu)$), tienen una dimensión de $(3, 13, 1)$, correspondientes a $k=3 clases, p=13 predictores$, y un valor determinado ($\mu$ y $(x-\mu)$, respectivamente)
  - `y` e `y**2`, representando los valores a predecir, con $k=3 clases, p=13 predictores, n=1 observaciones$

- En cuanto a los cambios realizados en el método `_predict_log_conditionals` (respecto a la implementación de `QDA_Chol3`)
  - Al calcular la diagonal de `L_inv`, debemos utilizar el método [diagonal](https://numpy-org.translate.goog/doc/2.2/reference/generated/numpy.diagonal.html?_x_tr_sl=en&_x_tr_tl=es&_x_tr_hl=es&_x_tr_pto=tc) especificando los ejes 1 y 2 (predictores y observaciones), para que el resultado sea un tensor con el valor de las diagonales.
    - Esto se logra de esta manera: `L_inv.diagonal(axis1=1, axis2=2)`, y nos devuelve una estructura de datos de $(3, 13)$ dimensiones
  - Al calcular el producto de `L_inv.diagonal(axis1=1, axis2=2)`, debemos especificar que se haga en el eje con 13 observaciones
    - Esto se logra mediante esta operación: `L_inv.diagonal(axis1=1, axis2=2).prod(axis=1)`
  - Al calcular la suma de $y**2$, el cual tiene dimensión $(3, 13, 1$), debemos indicarle también que debe realizarlo para los ejes 1 y 2, de modo que quede como resultado una estructura de datos de la forma $(3, )$
    - Esto se logra mediante la operación `(y**2).sum(axis=tuple([1,2]))`
  - De esta forma, tenemos como resultado del método `_predict_log_conditionals` una estructura de datos de la forma $(3, 1)$, en donde, para cada clase, tenemos el valor de *logaritmo de la probabilidad a posteriori*


##### Código para la resolución del punto 4.12

In [644]:
class TensorizedChol(QDA_Chol3):
    def _fit_params(self, X, y):
        # Utilizamos el método _fit_params de QDA_Chol3
        super()._fit_params(X,y)

        # stack onto new dimension
        self.tensor_L_inv = np.stack(self.L_invs)
        self.tensor_means = np.stack(self.means)

    def _predict_log_conditionals(self, x):
        L_inv = self.tensor_L_inv
        unbiased_x =  x - self.tensor_means

        y = L_inv @ unbiased_x

        # Necesitamos que ambos términos tengan dimensión (3,)
        return np.log(L_inv.diagonal(axis1=1, axis2=2).prod(axis=1)) -0.5 * (y**2).sum(axis=tuple([1,2]))
    
    def _predict_one(self, x):
        return np.argmax(self.log_a_priori + self._predict_log_conditionals(x))

In [645]:
# Definimos las variantes de QDA sobre las que vamos a hacer benchmarking
to_bench_4_12 = [
    TensorizedChol
]

# Definimos un benchmark por cada dataset
bench_4_12_wines = Benchmark(
    X_full_wine, y_full_encoded_wine,
    n_runs = 100,
    warmup = 20,
    mem_runs = 20,
    test_sz = 0.3,
    same_splits = False
)

bench_4_12_iris = Benchmark(
    X_full_iris, y_full_encoded_iris,
    n_runs = 100,
    warmup = 20,
    mem_runs = 20,
    test_sz = 0.3,
    same_splits = False
)

bench_4_12_penguins = Benchmark(
    X_full_penguins, y_full_encoded_penguins,
    n_runs = 100,
    warmup = 20,
    mem_runs = 20,
    test_sz = 0.3,
    same_splits = False
)

for model in to_bench_4_12:
    bench_4_12_wines.bench(model)
    bench_4_12_iris.bench(model)
    bench_4_12_penguins.bench(model)

Benching params:
Total runs: 140
Warmup runs: 20
Peak Memory usage runs: 20
Running time runs: 100
Train size rows (approx): 125
Test size rows (approx): 53
Test size fraction: 0.3
Benching params:
Total runs: 140
Warmup runs: 20
Peak Memory usage runs: 20
Running time runs: 100
Train size rows (approx): 105
Test size rows (approx): 45
Test size fraction: 0.3
Benching params:
Total runs: 140
Warmup runs: 20
Peak Memory usage runs: 20
Running time runs: 100
Train size rows (approx): 240
Test size rows (approx): 102
Test size fraction: 0.3


TensorizedChol (MEM):   0%|          | 0/20 [00:00<?, ?it/s]

TensorizedChol (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

TensorizedChol (MEM):   0%|          | 0/20 [00:00<?, ?it/s]

TensorizedChol (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

TensorizedChol (MEM):   0%|          | 0/20 [00:00<?, ?it/s]

TensorizedChol (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

In [646]:
bench_4_12_wines_summ = bench_4_12_wines.summary()[['train_median_ms', 'test_median_ms','mean_accuracy']]
bench_4_12_iris_summ = bench_4_12_iris.summary()[['train_median_ms', 'test_median_ms','mean_accuracy']]
bench_4_12_penguins_summ = bench_4_12_penguins.summary()[['train_median_ms', 'test_median_ms','mean_accuracy']]

bench_4_12_comparisson = pd.concat([bench_4_12_wines_summ, bench_4_12_iris_summ, bench_4_12_penguins_summ], ignore_index=False, axis=1)

display(bench_4_12_comparisson)

,train_median_ms,test_median_ms,mean_accuracy,train_median_ms,test_median_ms,mean_accuracy,train_median_ms,test_median_ms,mean_accuracy
model,,,,,,,,,
TensorizedChol,6.264,17.45705,0.982407,4.57805,11.7077,0.970667,3.71045,18.0712,0.986408


#### Respuesta Punto 4.13 - Implementar el modelo `EfficientChol` combinando los insights de `EfficientQDA` y `TensorizedChol`. Si se desea, se puede implementar `FasterChol` como ayuda, pero no se contempla para el punto.

##### Explicación del código para Punto 4.13

La implementación de `EfficientChol` combina las siguientes insights de `EfficientQDA` y `TensorizedChol`:
- La posibilidad de utilizar la diagonal de la matriz $L^{-1}$ para calcular el primer término del return, tal como hace `TensorizedChol`, evitando calcular determinantes
- La posibilidad de utilizar el producto elemento a elemento de dos matrices $\in \mathbb{R}^{n \times p}$, y luego calcular la suma columna por columna, tal como hace `EfficientQDA`
  - Esta matriz $A \in \mathbb{R}^{n \times p}$ corresponde a la traspuesta $A^{T} \in \mathbb{R}^{p \times n}$, la cual se obtiene a partir de la matriz (técnicamente, tensor) $L^{-1}$ multiplicada por el término $X - \bar{x}$, ambos términos utilizados en el método `TensorizedChol`

##### Código para la resolución del punto 4.13

In [672]:
class EfficientChol(TensorizedChol):
    def _fit_params(self, X, y):
        # Utilizamos el método _fit_params de QDA_Chol3
        super()._fit_params(X,y)

    def _predict_log_conditionals(self, X):
        unbiased_x =  X - self.tensor_means
        # pxn matrix
        y_p_x_n = self.tensor_L_inv @ unbiased_x
        # nxp matrix (la que necesitamos para implementar los insights de EfficientQDA)
        y_n_x_p = y_p_x_n.transpose(0,2,1)
        y = np.sum(y_n_x_p * y_n_x_p, axis=2)

        return np.log(self.tensor_L_inv.diagonal(axis1=1, axis2=2).prod(axis=1)).reshape(-1,1) -0.5 * (y) #(y**2).sum(axis=tuple([1,2]))
    
    def predict(self, X):
        return np.argmax(self.log_a_priori.reshape(-1,1) + self._predict_log_conditionals(X), axis=0)    

In [673]:
to_bench_4_13 = [
    EfficientChol
]

# Definimos un benchmark por cada dataset
bench_4_13_wines = Benchmark(
    X_full_wine, y_full_encoded_wine,
    n_runs = 100,
    warmup = 20,
    mem_runs = 20,
    test_sz = 0.3,
    same_splits = False
)

bench_4_13_iris = Benchmark(
    X_full_iris, y_full_encoded_iris,
    n_runs = 100,
    warmup = 20,
    mem_runs = 20,
    test_sz = 0.3,
    same_splits = False
)

bench_4_13_penguins = Benchmark(
    X_full_penguins, y_full_encoded_penguins,
    n_runs = 100,
    warmup = 20,
    mem_runs = 20,
    test_sz = 0.3,
    same_splits = False
)

for model in to_bench_4_13:
    bench_4_13_wines.bench(model)
    bench_4_13_iris.bench(model)
    bench_4_13_penguins.bench(model)

Benching params:
Total runs: 140
Warmup runs: 20
Peak Memory usage runs: 20
Running time runs: 100
Train size rows (approx): 125
Test size rows (approx): 53
Test size fraction: 0.3
Benching params:
Total runs: 140
Warmup runs: 20
Peak Memory usage runs: 20
Running time runs: 100
Train size rows (approx): 105
Test size rows (approx): 45
Test size fraction: 0.3
Benching params:
Total runs: 140
Warmup runs: 20
Peak Memory usage runs: 20
Running time runs: 100
Train size rows (approx): 240
Test size rows (approx): 102
Test size fraction: 0.3


EfficientChol (MEM):   0%|          | 0/20 [00:00<?, ?it/s]

EfficientChol (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

EfficientChol (MEM):   0%|          | 0/20 [00:00<?, ?it/s]

EfficientChol (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

EfficientChol (MEM):   0%|          | 0/20 [00:00<?, ?it/s]

EfficientChol (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

In [674]:
bench_4_13_wines_summ = bench_4_13_wines.summary()[['train_median_ms', 'test_median_ms','mean_accuracy']]
bench_4_13_iris_summ = bench_4_13_iris.summary()[['train_median_ms', 'test_median_ms','mean_accuracy']]
bench_4_13_penguins_summ = bench_4_13_penguins.summary()[['train_median_ms', 'test_median_ms','mean_accuracy']]

bench_4_13_comparisson = pd.concat([
    bench_4_13_wines_summ,
    bench_4_13_iris_summ,
    bench_4_13_penguins_summ
], ignore_index=False, axis=1)

display(bench_4_13_comparisson)

,train_median_ms,test_median_ms,mean_accuracy,train_median_ms,test_median_ms,mean_accuracy,train_median_ms,test_median_ms,mean_accuracy
model,,,,,,,,,
EfficientChol,6.2662,0.70165,0.982407,4.81045,0.55295,0.970667,4.9209,0.6099,0.986408


#### Respuesta Punto 4.14 - Comparar la performance de las 9 variantes de QDA implementadas ¿Qué se observa? A modo de opinión ¿Se condice con lo esperado?

En la siguiente tabla, se pueden observar la comparación de performance de los métodos `QDA`, `TensorizedQDA`, `FasterQDA`, `EfficientQDA`, `QDA_Chol1`, `QDA_Chol2`, `QDA_Chol3`, `TensorizedChol`, `EfficientChol`, considerando la siguiente configuración de benchmark:

- *Datasets Utilizados*:
  - Wine dataset (178 observaciones, 13 features)
  - Iris dataset (150 observaciones, 4 features)
  - Penguins dataset (342 observaciones, 4 features)
- *N runs*: 100
- *Warmup*: 20
- *Mem runs*: 20
- *Test size*: 30%
- *Same splits*: False

| | `Wines` - train median ms | `Wines` - test median ms | `Wines` - mean accuracy | `Iris` - train median ms | `Iris` - test median ms | `Iris` - mean accuracy | `Penguins` - train median ms | `Penguins` - test median ms | `Penguins` - mean accuracy |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| QDA | 3.98780 | 33.25830 | 0.982407 | 4.25995 | 34.57780 | 0.970667 | 5.58575 | 78.26470 | 0.986408 |
| TensorizedQDA | 4.83705 | 18.17975 | 0.982593 | 4.98060 | 15.95440 | 0.971111 | 5.07515 | 31.31465 | 0.987379 |
| FasterQDA | 4.09770 | 1.53885 | 0.985741 | 4.93395 | 0.62745 | 0.972889 | 3.44100 | 0.88935 | 0.986699 |
| EfficientQDA | 4.42700 | 0.67910 | 0.983333 | 4.93170 | 0.65105 | 0.972222 | 4.51895 | 0.67550 | 0.988252 |
| QDA_Chol1 | 5.61140 | 28.80490 | 0.986111 | 4.92075 | 23.95750 | 0.977333 | 4.94345 | 48.50960 | 0.987282 |
| QDA_Chol2 | 5.87675 | 72.22675 | 0.982222 | 4.30870 | 57.00105 | 0.972667 | 5.54490 | 107.53845 | 0.986699 |
| QDA_Chol3 | 3.48985 | 21.10900 | 0.984444 | 3.69155 | 20.08385 | 0.972222 | 3.64275 | 42.22700 | 0.987282 |
| TensorizedChol | 5.14410 | 13.31550 | 0.986667 | 4.76595 | 11.55610 | 0.971556 | 5.05445 | 27.64110 | 0.986893 |
| EfficientChol | 5.17625 | 0.74650 | 0.985556 | 4.66780 | 0.56245 | 0.974889 | 4.64620 | 0.58595 | 0.986505 |

Basándonos en estas pruebas, podemos afirmar que `TensorizedChol` y `EfficientChol` son, en términos de tiempos de ejecución, mejores que `QDA`, `TensorizedQDA` y las tres implementaciones de `QDA_Chol`, por lo menos respecto a tiempos medios en *test*. Esto se debe a que `TensorizedChol` y `EfficientChol` utilizan los insights de diversos métodos:
- tensorización, por parte de `TensorizedQDA`
- manejo producto elemento a elemento de dos matrices $A, B \in \mathbb{R}^{n \times p}$, por parte de `EfficientQDA`

Comparando `EfficientChol` con `FasterQDA` y `EfficientQDA` notamos que es mejor respecto a los tiempos medios en *test* en todos los datasets, salvo para el dataset de `Wines`, donde solamente mejora al tiempo de `FasterQDA`. En términos de `accuracy`, sucede que es mejor que `FasterQDA` y `EfficientQDA` en el dataset de `Iris`, y solamente supera a `EfficientQDA` para el dataset de `Wines`, mientras que 

Analizando los tiempos medios en *test* y el `accuracy` para `EfficientChol`, `FasterQDA` y `EfficientQDA` notamos lo siguiente:
- Para el dataset de `Iris` (150 observaciones, 4 features), `EfficientChol` es mejor que `FasterQDA` y `EfficientQDA` en términos de tiempo medio en *test* y *accuracy*
- Para el resto de datasets (`Wine`: 178 obs, 13 features; `Penguins`: 342 obs, 4 features), sucede lo siguiente:
    - Cuando `EfficientChol` es mejor en tiempo medio en *test*, su *accuracy* no supera al de `FasterQDA` o `EfficientQDA`
        - Ejemplo: en `Penguins`, donde `EfficientChol` mejora en tiempos a `FasterQDA` y `EfficientQDA`, pero no en *accuracy*
    - Cuando `EfficientChol` es mejor en *accuracy*, su tiempo medio en *test* es mayor al de `FasterQDA` o `EfficientQDA`
        - Ejemplo: en `Wines`, `EfficientChol` tiene mayor *accuracy* que `EfficientQDA`, pero su tiempo medio en *test* es mayor

En conclusión, podemos decir que `EfficientChol` es uno de los mejores métodos que tenemos (entre los 9 analizados), ya que en general mejora en tiempos al resto, y para aquellos casos en donde no lo hace, mejorará en términos de accuracy.

##### Código para la resolución del Punto 4.14

In [676]:
# Definimos las variantes de QDA sobre las que vamos a hacer benchmarking
to_bench_4_14 = [QDA, TensorizedQDA, FasterQDA, EfficientQDA, QDA_Chol1, QDA_Chol2, QDA_Chol3, TensorizedChol, EfficientChol]

bench_4_14_wines = Benchmark(
    X_full_wine, y_full_encoded_wine,
    n_runs = 100,
    warmup = 20,
    mem_runs = 20,
    test_sz = 0.3,
    same_splits = False
)

bench_4_14_iris = Benchmark(
    X_full_iris, y_full_encoded_iris,
    n_runs = 100,
    warmup = 20,
    mem_runs = 20,
    test_sz = 0.3,
    same_splits = False
)

bench_4_14_penguins = Benchmark(
    X_full_penguins, y_full_encoded_penguins,
    n_runs = 100,
    warmup = 20,
    mem_runs = 20,
    test_sz = 0.3,
    same_splits = False
)

Benching params:
Total runs: 140
Warmup runs: 20
Peak Memory usage runs: 20
Running time runs: 100
Train size rows (approx): 125
Test size rows (approx): 53
Test size fraction: 0.3
Benching params:
Total runs: 140
Warmup runs: 20
Peak Memory usage runs: 20
Running time runs: 100
Train size rows (approx): 105
Test size rows (approx): 45
Test size fraction: 0.3
Benching params:
Total runs: 140
Warmup runs: 20
Peak Memory usage runs: 20
Running time runs: 100
Train size rows (approx): 240
Test size rows (approx): 102
Test size fraction: 0.3


In [677]:
for model in to_bench_4_14:
    bench_4_14_wines.bench(model)
    bench_4_14_iris.bench(model)
    bench_4_14_penguins.bench(model)

QDA (MEM):   0%|          | 0/20 [00:00<?, ?it/s]

QDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

QDA (MEM):   0%|          | 0/20 [00:00<?, ?it/s]

QDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

QDA (MEM):   0%|          | 0/20 [00:00<?, ?it/s]

...

EfficientChol (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

EfficientChol (MEM):   0%|          | 0/20 [00:00<?, ?it/s]

EfficientChol (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

EfficientChol (MEM):   0%|          | 0/20 [00:00<?, ?it/s]

EfficientChol (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

In [678]:
bench_4_14_wines_summ = bench_4_14_wines.summary()[['train_median_ms', 'test_median_ms','mean_accuracy']]
bench_4_14_iris_summ = bench_4_14_iris.summary()[['train_median_ms', 'test_median_ms','mean_accuracy']]
bench_4_14_penguins_summ = bench_4_14_penguins.summary()[['train_median_ms', 'test_median_ms','mean_accuracy']]

bench_4_14_wines_summ.columns = "Wines__" + bench_4_14_wines_summ.columns
bench_4_14_iris_summ.columns = "Iris__" + bench_4_14_iris_summ.columns
bench_4_14_penguins_summ.columns = "Penguins__" + bench_4_14_penguins_summ.columns

bench_4_14_comparisson = pd.concat([bench_4_14_wines_summ, bench_4_14_iris_summ, bench_4_14_penguins_summ], ignore_index=False, axis=1)

In [679]:
display(bench_4_14_comparisson)

,Wines__train_median_ms,Wines__test_median_ms,Wines__mean_accuracy,Iris__train_median_ms,Iris__test_median_ms,Iris__mean_accuracy,Penguins__train_median_ms,Penguins__test_median_ms,Penguins__mean_accuracy
model,,,,,,,,,
QDA,3.98780,33.25830,0.982407,4.25995,34.57780,0.970667,5.58575,78.26470,0.986408
TensorizedQDA,4.83705,18.17975,0.982593,4.98060,15.95440,0.971111,5.07515,31.31465,0.987379
FasterQDA,4.09770,1.53885,0.985741,4.93395,0.62745,0.972889,3.44100,0.88935,0.986699
EfficientQDA,4.42700,0.67910,0.983333,4.93170,0.65105,0.972222,4.51895,0.67550,0.988252
QDA_Chol1,5.61140,28.80490,0.986111,4.92075,23.95750,0.977333,4.94345,48.50960,0.987282
QDA_Chol2,5.87675,72.22675,0.982222,4.30870,57.00105,0.972667,5.54490,107.53845,0.986699
QDA_Chol3,3.48985,21.10900,0.984444,3.69155,20.08385,0.972222,3.64275,42.22700,0.987282
TensorizedChol,5.14410,13.31550,0.986667,4.76595,11.55610,0.971556,5.05445,27.64110,0.986893
EfficientChol,5.17625,0.74650,0.985556,4.66780,0.56245,0.974889,4.64620,0.58595,0.986505
